# Consumo del modelo y app Gradio — PhysioVision Ex1

**PhysioVision · Diplomado Modulo 5 · notebook 2 de 2: consumo**

El primer notebook ([`modelo_xgboost_ex1.ipynb`](modelo_xgboost_ex1.ipynb)) entrena y
resguarda. Este **no entrena nada**: carga los artefactos de `models/` y los usa, hasta
levantar la aplicacion.

| # | Seccion | Que hace |
|---|---|---|
| 1 | Configuracion | rutas y comprobacion de que los artefactos existen |
| 2 | Carga de los artefactos | el modelo y su contrato de variables |
| 3 | Datos de referencia | `df_reps` desde `data/datasets/` (para las demos) |
| 4 | **Evaluacion en tiempo real** | camara -> repeticiones -> clase, con veredicto al cerrarse cada repeticion |
| 5 | Redaccion con Gemini | la consigna, dicha como una persona |
| 6 | Voz con Text-to-Speech | la consigna, en audio |
| 7 | **PhysioVision en Gradio** | la app, en su forma mas simple |

**Requisito**: `models/xgboost_model.json` y `models/feature_contract.json`. Los genera
la seccion 13 del primer notebook. Si no estan, la celda 1 lo dice y para.

El notebook es **autocontenido**: no importa ni una linea del codigo de la
aplicacion. Las piezas que necesita —geometria, variables, segmentacion causal,
detector, redactor y voz— estan definidas en sus propias celdas, copiadas de las
de la app, para que se abra suelto en Google Colab.

Lo que hace cada pieza, y lo que no:

| Pieza | Decide |
|---|---|
| XGBoost | la clase de la repeticion |
| `knowledge_base/ejercicios.json` | que recomendar para esa clase |
| Gemini | como decirlo |
| Cloud TTS | como suena |

> **Descargo clinico.** Material academico. No constituye diagnostico ni sustituye el
> criterio de un profesional de la salud.

---
## 0. Entorno y reproducibilidad

Igual que el notebook de entrenamiento, esta libreta corre en local y en **Google
Colab**. La celda 0.1 trae el código, instala lo que falte y comprueba qué hay
disponible.

Lo esencial **viaja con el repositorio**: el modelo, su contrato de variables y
los umbrales están versionados, así que las secciones 2, 4, 5, 6 y 7 —carga,
evaluación en vivo, redacción, voz e interfaz— funcionan en Colab tal cual.

Lo que no viaja son los datos de los pacientes:

| Sección | Necesita | Sin ello |
|---|---|---|
| 3 y los ejemplos de 5 | dataset de repeticiones | se saltan con aviso |
| 2, 4, 6 y 7 | solo `models/` y claves opcionales | funcionan siempre |

Las secciones 4 y 7 piden **cámara**: los fotogramas salen de tu navegador, no
de ningún archivo. No hay ninguna evaluación sobre vídeo grabado en este
notebook — la unidad de trabajo es la repetición, en el instante en que ocurre.

**Convención de este notebook**: las celdas que dependen de datos empiezan con
`if not HAY_...`, avisan y siguen. Ninguna interrumpe la ejecución.

En Colab la interfaz de la sección 7 se abre con un enlace público temporal
(`share=True`), porque no hay navegador local al que asomarse.

In [1]:
# 0.1 Entorno: codigo, dependencias y datos disponibles.
#     Idempotente y sin magias de Jupyter, para que valga igual en Colab, en
#     Jupyter local y ejecutado como script.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/daff-250900/phyisiovision.git"

#: Lo que Colab no trae. Versiones del entorno donde se valido el modelo
#: (requirements.lock): mediapipe fija los landmarks y xgboost el clasificador.
PAQUETES = {"mediapipe": "mediapipe==0.10.35", "xgboost": "xgboost==3.2.0",
            "gradio": "gradio==6.20.0", "google.genai": "google-genai==2.14.0"}

EN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

if EN_COLAB:
    destino = Path("/content/phyisiovision")
    if not destino.exists():
        print("Clonando el repositorio...")
        subprocess.run(["git", "clone", "--depth", "1", "--quiet", REPO, str(destino)],
                       check=True)
    os.chdir(destino / "entrenamiento")

    faltan = [pin for modulo, pin in PAQUETES.items()
              if importlib.util.find_spec(modulo.split(".")[0]) is None]
    if faltan:
        print(f"Instalando {', '.join(faltan)} (un par de minutos)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *faltan],
                       check=True)

# El repositorio se clona por los artefactos (models/, knowledge_base/), no por
# el codigo: este notebook no importa nada de la aplicacion y no toca sys.path.
# Todo lo que necesita esta definido en sus propias celdas.
NB_DIR = Path.cwd()
BASE_DIR = NB_DIR.parent if NB_DIR.name == "entrenamiento" else NB_DIR
if not (BASE_DIR / "models").exists() and (BASE_DIR.parent / "models").exists():
    BASE_DIR = BASE_DIR.parent

# Los datos pueden estar en el repositorio o montados aparte (Drive, un disco).
DATOS_DIR = Path(os.environ.get("PHYSIOVISION_DATOS_EXTERNOS", BASE_DIR / "data"))
DATASET_DIR = DATOS_DIR / "datasets"
MODEL_DIR = BASE_DIR / "models"

HAY_DATASET = (DATASET_DIR / "ex1_repeticiones.csv").exists()
HAY_MODELO = (MODEL_DIR / "xgboost_model.json").exists()

print(f"Entorno  : {'Google Colab' if EN_COLAB else 'local'}")
print(f"Proyecto : {BASE_DIR}")
print()
for etiqueta, hay, secciones in [
    ("modelo y contrato", HAY_MODELO, "2, 4, 5, 6 y 7 (lo esencial)"),
    ("dataset de repeticiones", HAY_DATASET, "3 y los ejemplos de 5"),
]:
    print(f"  {'si' if hay else 'NO':3s}  {etiqueta:26s} {secciones}")
print(f"  --   camara del navegador     4 y 7 (evaluacion en tiempo real)")

if not HAY_MODELO:
    print()
    print("Sin modelo entrenado: ejecuta antes modelo_xgboost_ex1.ipynb, o clona")
    print("el repositorio completo, donde models/ va versionado.")


Entorno  : local
Proyecto : /Users/dafnezepedagonzalez/Documents/Diplomado/Modulo5/physiovision_gradio

  si   modelo y contrato          2, 4, 5, 6 y 7 (lo esencial)
  si   dataset de repeticiones    3 y los ejemplos de 5
  --   camara del navegador     4 y 7 (evaluacion en tiempo real)


---
## 1. Configuracion

Solo rutas: ni semillas de entrenamiento ni parametros del pipeline, porque aqui no se
reentrena nada. Los umbrales de segmentacion y de las reglas se leen de
`models/umbrales_ex1.json`, que exporto el primer notebook, para que la inferencia use
exactamente los mismos que el entrenamiento.

La celda **avisa en grande si faltan los artefactos** y las secciones que los
necesitan se saltan. Es preferible eso a que el clasificador caiga en silencio a
las reglas biomecanicas y el resto del notebook parezca funcionar sin usar el
modelo.

In [2]:
from __future__ import annotations

import json
import os
import sys
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# ---- Ejercicio y artefactos -------------------------------------------------
# Las rutas y la deteccion de entorno vienen de la seccion 0.1.
# Dos identificadores distintos del mismo ejercicio, y conviene no confundirlos:
EJERCICIO = "Ex1"                          # carpeta de datos y sufijo de artefactos
EJERCICIO_KB = "elevacion_lateral_hombro"  # clave en knowledge_base/ejercicios.json

RUTA_MODELO = MODEL_DIR / "xgboost_model.json"
RUTA_CONTRATO = MODEL_DIR / "feature_contract.json"

FPS_NOMINAL = 30.0
# El entrenamiento uso la variante "heavy" (proceso por lotes, primaba la precision).
# Aqui hay una persona esperando delante de la pantalla, asi que "full".
VARIANTE_MP = os.environ.get("PV_VARIANTE_MP", "full")

# ---- Camara: lo que sube y lo que baja --------------------------------------
# Sin `constraints` el navegador abre la camara a su tamano nativo y cada
# fotograma viaja entero. Medido sobre un fotograma real de Ex1: 1280x720 son
# 121 KB de base64 por fotograma y 640x480 son 52 KB. Y la tasa no la fija
# `stream_every` sino el viaje de ida y vuelta —el frontend de Gradio no captura
# el siguiente hasta que vuelve el anterior—, asi que bajar la resolucion es la
# palanca que sube los fps. 640x480 no empeora la deteccion: MediaPipe recorta y
# reescala a 256x256 para la pose.
RESOLUCION_CAMARA = {
    "video": {
        "width": {"ideal": int(os.environ.get("PV_CAMARA_ANCHO", "640"))},
        "height": {"ideal": int(os.environ.get("PV_CAMARA_ALTO", "480"))},
    }
}

# Ancho de la imagen anotada que se devuelve al navegador. Es una decision de
# coste, no de calidad: **solo afecta a lo que se ve**, porque las mediciones se
# hacen antes, sobre el fotograma completo, y el modelo no ve esta imagen.
ANCHO_SALIDA = int(os.environ.get("PV_ANCHO_SALIDA", "960"))

print(f"Ejercicio : {EJERCICIO}")
print(f"MediaPipe : variante {VARIANTE_MP}")
print(f"Camara    : {RESOLUCION_CAMARA['video']['width']['ideal']}"
      f"x{RESOLUCION_CAMARA['video']['height']['ideal']} pedidos al navegador"
      f" · salida a {ANCHO_SALIDA} px")

faltan = [p.name for p in (RUTA_MODELO, RUTA_CONTRATO) if not p.exists()]
if faltan:
    # No se lanza excepcion: el notebook tiene que poder leerse y ejecutarse
    # entero aunque falte el modelo. Las celdas que lo necesitan se saltan.
    print(f"Faltan artefactos en models/: {faltan}")
    print("Ejecuta antes modelo_xgboost_ex1.ipynb (seccion 13).")

print("Artefactos:")
for p in (RUTA_MODELO, RUTA_CONTRATO, MODEL_DIR / "umbrales_ex1.json"):
    estado = f"{p.stat().st_size / 1024:.0f} KB" if p.exists() else "AUSENTE (opcional)"
    print(f"  {p.name:24s} {estado}")

Ejercicio : Ex1
MediaPipe : variante full
Artefactos:
  xgboost_model.json       483 KB
  feature_contract.json    3 KB
  umbrales_ex1.json        0 KB


---
## 2. Carga de los artefactos entrenados

Lo unico delicado de esta parte es el orden de las columnas. La celda **no lo tiene
escrito a mano**: lo lee de `models/feature_contract.json`, que trae los nombres de
las variables en el orden exacto que vio el modelo al entrenar, el mapa de clases y
las medianas con las que imputar lo que falte.

Con eso y el `.json` del modelo se clasifica sin conocer una sola linea del codigo de
la aplicacion — que es justo lo que se comprueba aqui. Si el modelo y el contrato no
coincidieran, la celda para con un `AssertionError` en lugar de predecir en silencio
sobre columnas desalineadas, que es el fallo mas caro posible aqui.

La app hace exactamente esto mismo, con una diferencia: si el modelo no esta, no
falla, cae a las reglas biomecanicas y lo declara en `source: "reglas"`. Por eso el
resultado de `clasificar()` lleva tambien ese campo.


In [3]:
# Requiere el modelo entrenado (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_MODELO:
    print("Se salta: falta el modelo entrenado.")
else:
    # -----------------------------------------------------------------------
    # El contrato es lo unico que hace falta saber del modelo
    #
    # `models/feature_contract.json` trae los nombres de las variables EN ORDEN,
    # el mapa de clases y las medianas con las que imputar lo que falte. Con eso
    # y el .json del modelo se clasifica sin conocer el codigo de la aplicacion,
    # que es justo lo que se comprueba aqui.
    # -----------------------------------------------------------------------
    from xgboost import XGBClassifier

    CONTRATO = json.loads(RUTA_CONTRATO.read_text(encoding="utf-8"))
    FEATURES = CONTRATO["feature_names"]
    CLASES = {int(k): v for k, v in CONTRATO["labels"].items()}
    CLASE_A_ID = {v: k for k, v in CLASES.items()}
    MEDIANAS = CONTRATO["imputacion_medianas"]

    MODELO = XGBClassifier()
    MODELO.load_model(RUTA_MODELO)
    assert list(MODELO.feature_names_in_) == FEATURES, (
        "El modelo y el contrato no coinciden en el orden de las variables")

    def clasificar(variables: dict) -> dict:
        """Clasifica una repeticion. Ordena e imputa segun el contrato."""
        fila = {c: float(variables.get(c, np.nan)) for c in FEATURES}
        vector = pd.DataFrame([fila]).fillna(pd.Series(MEDIANAS))
        probabilidades = MODELO.predict_proba(vector)[0]
        clase = int(probabilidades.argmax())
        return {"class_id": clase, "label": CLASES[clase],
                "confidence": float(probabilidades.max()),
                "probabilities": {CLASES[i]: float(p)
                                  for i, p in enumerate(probabilidades)},
                # La app declara aqui si predijo el modelo o las reglas de
                # respaldo. Este notebook exige el modelo, asi que siempre
                # es "xgboost": comprobarlo distingue "funciona" de
                # "funciona sin usar el modelo".
                "source": "xgboost"}

    # -----------------------------------------------------------------------
    # Base de conocimiento: que recomendar para cada clase
    # -----------------------------------------------------------------------
    CONOCIMIENTO = json.loads(
        (BASE_DIR / "knowledge_base" / "ejercicios.json").read_text("utf-8"))

    def recomendacion(clase: str, ejercicio: str = EJERCICIO_KB) -> dict:
        """Ficha completa de una clase: titulo, consignas, recomendacion."""
        return CONOCIMIENTO.get(ejercicio, {}).get("errores", {}).get(clase, {
            "titulo": "Recomendacion general", "consignas": ["Despacio y controlado"],
            "recomendacion": "Realiza el movimiento de forma lenta y controlada.",
            "precaucion": "Deten el ejercicio ante dolor agudo."})

    def consigna(clase: str, indice: int = 0, ejercicio: str = EJERCICIO_KB) -> str:
        """Consigna corta para decir en voz alta. Rota entre las disponibles."""
        opciones = recomendacion(clase, ejercicio).get("consignas") or []
        return (opciones[indice % len(opciones)] if opciones
                else recomendacion(clase, ejercicio)["recomendacion"].split(".")[0])

    OBJETIVOS = CONOCIMIENTO.get(EJERCICIO_KB, {}).get("objetivos", {})
    REPS_OBJETIVO = int(CONOCIMIENTO.get(EJERCICIO_KB, {}).get(
        "repeticiones_objetivo", 0))

    print(f"contrato    : v{CONTRATO['version']}, creado {CONTRATO['creado'][:10]}")
    print(f"variables   : {len(FEATURES)}")
    print(f"clases      : {CLASES}")
    print(f"macro-F1    : {CONTRATO['metricas_loso']['macro_f1']:.3f} "
          f"IC95% {CONTRATO['ic95_macro_f1']}")
    print(f"etiquetado  : {CONTRATO['entrenamiento']['etiquetado']}")
    print(f"\nconocimiento: {len(CONOCIMIENTO)} ejercicio(s); "
          f"serie de {REPS_OBJETIVO} repeticiones")
    print(f"objetivos   : {[k for k in OBJETIVOS if k != 'fuente']}")

contrato    : v1.0.0, creado 2026-08-01
variables   : 26
clases      : {0: 'rango_insuficiente', 1: 'correcto', 2: 'compensacion_tronco'}
macro-F1    : 0.869 IC95% [0.7912, 0.9234]
etiquetado  : manual_revisado

conocimiento: 1 ejercicio(s); serie de 5 repeticiones
objetivos   : ['rom', 'tronco', 'duracion_s']


---
## 3. Datos de referencia desde disco

No hace falta reextraer landmarks ni reentrenar: el primer notebook dejo las
repeticiones y sus etiquetas en `data/datasets/`. Se reconstruye `df_reps` para tener
repeticiones reales con las que probar la redaccion (seccion 5) sin depender de
procesar un video.

La etiqueta se guarda **por gesto** —un gesto es la misma repeticion vista por las dos
camaras—, asi que hay que reunirla con las filas por repeticion.

In [4]:
# Requiere modelo y dataset (seccion 0). Sin ellos se salta y el notebook sigue.
if not (HAY_DATASET and HAY_MODELO):
    print("Se salta: hacen falta el modelo y el dataset de repeticiones.")
else:
    def cargar_repeticiones() -> pd.DataFrame:
        reps = pd.read_csv(DATASET_DIR / "ex1_repeticiones.csv")
        etiquetas = pd.read_csv(DATASET_DIR / "ex1_etiquetas.csv")
        df = reps.merge(etiquetas[["gesto_id", "etiqueta"]], on="gesto_id", how="left")
        sin_etiqueta = int(df.etiqueta.isna().sum())
        if sin_etiqueta:
            print(f"AVISO: {sin_etiqueta} repeticiones sin etiqueta, se descartan.")
            df = df[df.etiqueta.notna()]
        return df.reset_index(drop=True)


    df_reps = cargar_repeticiones()
    df_reps["y"] = df_reps.etiqueta.map(CLASE_A_ID)
    print(f"{len(df_reps)} repeticiones, {df_reps.sujeto.nunique()} sujetos, "
          f"{df_reps.gesto_id.nunique()} gestos")
    print(df_reps.etiqueta.value_counts().to_string())

    # Prueba de humo del contrato: una repeticion real pasada por el clasificador.
    # Si `source` no es "xgboost", el modelo no se cargo y todo lo que sigue seria
    # el comportamiento degradado, no el del modelo entrenado.
    ejemplo = clasificar(df_reps.iloc[0].to_dict())
    print(f"\nPrimera repeticion ({df_reps.iloc[0].sujeto}, etiqueta "
          f"{df_reps.iloc[0].etiqueta}):")
    for k, v in ejemplo.items():
        print(f"  {k}: {v}")
    assert ejemplo["source"] == "xgboost"

334 repeticiones, 13 sujetos, 167 gestos
etiqueta
correcto               186
compensacion_tronco    134
rango_insuficiente      14

Primera repeticion (PM_000, etiqueta compensacion_tronco):
  class_id: 2
  label: compensacion_tronco
  confidence: 0.91883784532547
  probabilities: {'rango_insuficiente': 0.004112705588340759, 'correcto': 0.07704946398735046, 'compensacion_tronco': 0.91883784532547}
  source: xgboost


---
## 4. Evaluacion en tiempo real, repeticion a repeticion

Aqui esta la diferencia con el entrenamiento. Alli se procesaba el video entero
y se segmentaba mirando la senal completa: se sabe donde estan los picos porque
ya han pasado todos. **Con una camara eso no existe.** Los fotogramas llegan de
uno en uno y hay que decidir sin ver el futuro.

Por eso en este notebook no se evalua ningun video grabado: la unidad de trabajo
es la repeticion, y su veredicto tiene que llegar **en el instante en que se
cierra**, mientras el paciente sigue delante de la camara. Un informe al final de
la serie llega tarde para corregir nada.

Tres piezas resuelven esa diferencia, y son las mismas que usa la aplicacion:

| Pieza | Que hace |
|---|---|
| `BufferCausal` | Suaviza con un retardo fijo: para filtrar un instante hace falta media ventana de fotogramas posteriores, asi que se emite con ese retraso |
| `SegmentadorOnline` | Detecta valle → pico → valle segun van llegando, confirmando cada extremo con una prominencia minima |
| `AcumuladorEnVivo` | Une lo anterior con la deteccion del lado activo y produce, al cerrarse cada repeticion, el diccionario de variables listo para el modelo |

**La deteccion del lado no se puede hacer con una ventana fija.** Medido sobre
Ex1: a los 3 segundos ambos brazos presentan 75-92 grados de recorrido y la
razon entre ellos es de 1.03-1.07, indistinguible del ruido. Lo que discrimina
es esa razon, que tarda entre 5 y 12 segundos en superar el umbral. Por eso el
calentamiento no termina por tiempo sino cuando hay un ganador claro, y las
repeticiones ocurridas mientras tanto no se pierden: el bufer se reprocesa con
el lado ya decidido.

Todo el codigo que sigue esta definido en el propio notebook. Es una copia
deliberada del de la aplicacion: el modelo solo da los mismos numeros si las
variables se calculan exactamente igual que cuando se entreno.


In [5]:
# ---------------------------------------------------------------------------
# Geometria y variables: el mismo codigo que el notebook de entrenamiento
#
# Se repite aqui a proposito. Los dos notebooks tienen que poder abrirse
# sueltos en Colab, y el modelo solo da los mismos numeros si las variables
# se calculan exactamente igual que cuando se entreno.
# ---------------------------------------------------------------------------
from scipy.signal import find_peaks, savgol_filter

#: Índices de MediaPipe Pose relevantes para un ejercicio de miembro superior.
LANDMARKS = {
    0: "nose", 7: "left_ear", 8: "right_ear",
    11: "left_shoulder", 12: "right_shoulder",
    13: "left_elbow", 14: "right_elbow",
    15: "left_wrist", 16: "right_wrist",
    23: "left_hip", 24: "right_hip",
    25: "left_knee", 26: "right_knee",
}

NOMBRES = list(LANDMARKS.values())

#: Esquema de los CSV de `data/landmarks/`: normalizados (x, y, z, visibilidad)
#: seguidos de world (wx, wy, wz).
COLUMNAS = ["frame", "t_seg"]
for _n in NOMBRES:
    COLUMNAS += [f"{_n}_x", f"{_n}_y", f"{_n}_z", f"{_n}_v"]
for _n in NOMBRES:
    COLUMNAS += [f"{_n}_wx", f"{_n}_wy", f"{_n}_wz"]

FPS_NOMINAL = 30.0

print(f"{len(LANDMARKS)} landmarks -> {len(COLUMNAS)} columnas por CSV")
#: Vertical hacia arriba en el sistema de world landmarks (y crece hacia abajo).
ARRIBA = np.array([0.0, -1.0, 0.0])

def w(df: pd.DataFrame, nombre: str) -> np.ndarray:
    """Serie de coordenadas world `(n_frames, 3)` de un landmark."""
    return df[[f"{nombre}_wx", f"{nombre}_wy", f"{nombre}_wz"]].to_numpy(dtype=float)

def angulo_3d(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    """Ángulo ABC en grados, vectorizado sobre todos los frames."""
    ba, bc = a - b, c - b
    den = np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        cos = np.einsum("ij,ij->i", ba, bc) / den
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

def angulo_con_vertical(v: np.ndarray) -> np.ndarray:
    """Ángulo en grados entre cada vector y la vertical hacia arriba."""
    den = np.linalg.norm(v, axis=1)
    # Producto escalar elemento a elemento en vez de `v @ ARRIBA`: matmul pasa
    # por BLAS, que deja marcada la bandera de desbordamiento de coma flotante y
    # hace que `errstate` emita un RuntimeWarning espurio con datos perfectamente
    # normales. El resultado numérico es idéntico.
    with np.errstate(invalid="ignore", divide="ignore"):
        cos = (v * ARRIBA).sum(axis=1) / den
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))
#: Umbrales del pipeline. Si existe models/umbrales_ex1.json —que exporta la
#: seccion 13— se usan los suyos, para que la segmentacion en vivo aplique los
#: mismos cortes que el entrenamiento.
_UMBRALES = {"MIN_VISIBILIDAD": 0.5, "SUAVIZADO_SEG": 0.40, "MIN_DUR_REP": 1.0,
             "MIN_AMPLITUD_REP": 20.0, "MAX_DUR_REP": 20.0, "MAX_LADO": 960.0}
_ruta_umbrales = MODEL_DIR / "umbrales_ex1.json"
if _ruta_umbrales.exists():
    _UMBRALES.update({k: float(v) for k, v in
                      json.loads(_ruta_umbrales.read_text("utf-8")).items()
                      if k in _UMBRALES})

MIN_VISIBILIDAD = _UMBRALES["MIN_VISIBILIDAD"]
SUAVIZADO_SEG = _UMBRALES["SUAVIZADO_SEG"]
MIN_DUR_REP = _UMBRALES["MIN_DUR_REP"]
MIN_AMPLITUD_REP = _UMBRALES["MIN_AMPLITUD_REP"]
MAX_DUR_REP = _UMBRALES["MAX_DUR_REP"]
#: Lado mayor al que se reescala cada fotograma antes de inferir. Sale del
#: mismo JSON que el resto: cambiarlo cambia los landmarks, y por tanto las
#: variables que ve el modelo.
MAX_LADO = int(_UMBRALES["MAX_LADO"])
print("umbrales:", {k: round(v, 2) for k, v in _UMBRALES.items()})

def series_angulares(df: pd.DataFrame, lado: str) -> pd.DataFrame:
    """Convierte un DataFrame de landmarks en las series angulares de un lado.

    Args:
        df: filas = frames, columnas según `COLUMNAS`.
        lado: ``"left"`` o ``"right"``, el brazo que ejecuta el ejercicio.
    """
    otro = "right" if lado == "left" else "left"
    hombro, codo, muneca = w(df, f"{lado}_shoulder"), w(df, f"{lado}_elbow"), w(df, f"{lado}_wrist")
    cadera, oreja = w(df, f"{lado}_hip"), w(df, f"{lado}_ear")
    hombro_o, cadera_o, codo_o = w(df, f"{otro}_shoulder"), w(df, f"{otro}_hip"), w(df, f"{otro}_elbow")

    hombro_medio = (hombro + hombro_o) / 2
    cadera_media = (cadera + cadera_o) / 2
    tronco = hombro_medio - cadera_media
    ancho_hombros = np.linalg.norm(hombro - hombro_o, axis=1)
    largo_torso = np.linalg.norm(tronco, axis=1)

    # Inclinación lateral con signo: positiva = el tronco se inclina hacia el
    # lado contrario al brazo que trabaja, que es la compensación relevante.
    signo = 1.0 if lado == "left" else -1.0
    lean_lateral = signo * np.degrees(np.arctan2(tronco[:, 0], -tronco[:, 1]))

    out = pd.DataFrame({
        "frame": df["frame"].to_numpy(),
        "t_seg": df["t_seg"].to_numpy(),
        "abduccion_hombro": angulo_3d(cadera, hombro, codo),
        "flexion_codo": angulo_3d(hombro, codo, muneca),
        "abduccion_contralateral": angulo_3d(cadera_o, hombro_o, codo_o),
        "inclinacion_tronco": angulo_con_vertical(tronco),
        "lean_lateral": lean_lateral,
        # Elevación escapular: hombro que sube hacia la oreja. Se normaliza por
        # el ancho de hombros para que no dependa del tamaño del sujeto.
        "elevacion_escapular": -np.linalg.norm(hombro - oreja, axis=1) / np.where(
            ancho_hombros > 1e-6, ancho_hombros, np.nan),
        "largo_torso": largo_torso,
        "visibilidad": df[[f"{lado}_shoulder_v", f"{lado}_elbow_v", f"{lado}_wrist_v",
                           f"{lado}_hip_v"]].mean(axis=1).to_numpy(),
    })
    out["valido"] = out["abduccion_hombro"].notna() & (out["visibilidad"] >= MIN_VISIBILIDAD)
    return out

def detectar_lado(df: pd.DataFrame) -> tuple[str, float, float]:
    """Elige el brazo activo como el de mayor recorrido angular.

    El lado que ejecuta el ejercicio cambia entre sujetos, así que fijarlo por
    configuración deja variables sin señal en parte de la población.

    Returns:
        ``(lado, rango_izquierdo, rango_derecho)`` con los rangos en grados.
    """
    rangos = {}
    for lado in ("left", "right"):
        s = series_angulares(df, lado)
        v = s.loc[s.valido, "abduccion_hombro"]
        rangos[lado] = 0.0 if len(v) < 10 else float(
            np.percentile(v, 97.5) - np.percentile(v, 2.5))
    lado = max(rangos, key=rangos.get)
    return lado, rangos["left"], rangos["right"]

print(series_angulares.__doc__.splitlines()[0])
from scipy.signal import find_peaks, savgol_filter

def ventana_savgol(fps: float) -> int:
    """Ventana impar del filtro Savitzky-Golay para una tasa de frames dada."""
    return int(SUAVIZADO_SEG * fps) | 1

def suavizar(serie: pd.Series, fps: float) -> np.ndarray:
    """Interpola huecos cortos y suaviza con Savitzky-Golay.

    Savitzky-Golay preserva la amplitud de los picos; una media móvil los
    aplanaría y sesgaría el rango de movimiento a la baja.

    Nota: la ventana es **centrada**, por lo que este filtro no es causal. Para
    la ruta en vivo, ver `BufferCausal` unas celdas mas abajo.
    """
    v = serie.to_numpy(dtype=float).copy()
    s = pd.Series(v).interpolate(limit=5, limit_direction="both")
    ventana = ventana_savgol(fps)
    if len(s) <= ventana or ventana < 5:
        return s.to_numpy()
    return savgol_filter(s.to_numpy(), ventana, 3, mode="interp")

def prominencia_para(senal: np.ndarray) -> float:
    """Prominencia mínima de un pico, relativa al recorrido de la señal."""
    finita = senal[np.isfinite(senal)]
    if len(finita) == 0:
        return MIN_AMPLITUD_REP / 2
    rango = np.percentile(finita, 97.5) - np.percentile(finita, 2.5)
    return max(0.25 * rango, MIN_AMPLITUD_REP / 2)

def segmentar(senal: np.ndarray, fps: float) -> list[dict]:
    """Divide la señal de abducción en repeticiones: valle → pico → valle.

    Requiere la señal completa (`find_peaks` es global). La versión causal
    equivalente está en `SegmentadorOnline`, unas celdas más abajo.

    Returns:
        Lista de dicts con ``inicio``, ``pico``, ``fin`` (índices de frame),
        ``amplitud`` (grados) y ``duracion_s``.
    """
    finita = senal[np.isfinite(senal)]
    if len(finita) < int(3 * fps):
        return []
    rango = np.percentile(finita, 97.5) - np.percentile(finita, 2.5)
    if rango < MIN_AMPLITUD_REP:
        return []

    prominencia = max(0.25 * rango, MIN_AMPLITUD_REP / 2)
    distancia = int(MIN_DUR_REP * fps)
    picos, _ = find_peaks(senal, prominence=prominencia, distance=distancia)
    valles, _ = find_peaks(-senal, prominence=prominencia * 0.6, distance=distancia)
    if len(picos) == 0 or len(valles) < 2:
        return []

    reps = []
    for pico in picos:
        antes = valles[valles < pico]
        despues = valles[valles > pico]
        if len(antes) == 0 or len(despues) == 0:
            continue
        ini, fin = int(antes[-1]), int(despues[0])
        amplitud = senal[pico] - max(senal[ini], senal[fin])
        duracion = (fin - ini) / fps
        if amplitud < MIN_AMPLITUD_REP or duracion < MIN_DUR_REP or duracion > MAX_DUR_REP:
            continue
        reps.append({"inicio": ini, "pico": int(pico), "fin": fin,
                     "amplitud": float(amplitud), "duracion_s": float(duracion)})

    # Eliminar solapamientos conservando la repetición de mayor amplitud.
    reps.sort(key=lambda r: r["inicio"])
    limpias: list[dict] = []
    for r in reps:
        if limpias and r["inicio"] < limpias[-1]["fin"]:
            if r["amplitud"] > limpias[-1]["amplitud"]:
                limpias[-1] = r
        else:
            limpias.append(r)
    return limpias

print("suavizado y segmentacion definidos en el propio notebook")
def ldlj(velocidad: np.ndarray, dt: float) -> float:
    """*Log dimensionless jerk*: métrica de suavidad del movimiento.

    Más negativo = movimiento más brusco. Es invariante a la amplitud y a la
    duración, y es estándar en rehabilitación para medir control motor: capta
    descontrol que ningún umbral angular detecta.
    """
    v = velocidad[np.isfinite(velocidad)]
    if len(v) < 5:
        return np.nan
    T = len(v) * dt
    v_pico = np.abs(v).max()
    if v_pico < 1e-6 or T < 1e-6:
        return np.nan
    jerk = np.gradient(np.gradient(v, dt), dt)
    integrar = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    integral = integrar(jerk ** 2, dx=dt)
    valor = (T ** 3 / v_pico ** 2) * integral
    return float(-np.log(valor)) if valor > 0 else np.nan

def features_repeticion(s: pd.DataFrame, rep: dict, fps: float,
                        idx: int, n_reps: int) -> dict:
    """Variables de una repetición.

    Args:
        s: salida de `series_angulares` con la columna `abduccion_suave` añadida.
        rep: un elemento de `segmentar()`.
        fps: tasa de frames.
        idx: índice de la repetición dentro de la serie.
        n_reps: total de repeticiones de la serie.

    Returns:
        Dict de variables, o ``{}`` si la repetición no tiene frames válidos
        suficientes.
    """
    ini, pico, fin = rep["inicio"], rep["pico"], rep["fin"]
    seg = s.iloc[ini:fin + 1]
    val = seg[seg.valido]
    if len(val) < 5:
        return {}

    dt = 1.0 / fps
    ang = seg["abduccion_suave"].to_numpy(dtype=float)
    vel = np.gradient(ang, dt)                      # grados/s
    t_con = max(pico - ini, 1) * dt                 # fase concéntrica (subir)
    t_exc = max(fin - pico, 1) * dt                 # fase excéntrica (bajar)
    umbral_pico = val.abduccion_hombro.max() * 0.90

    return {
        # --- rango de movimiento
        "rom_max": float(val.abduccion_hombro.max()),
        "rom_min": float(val.abduccion_hombro.min()),
        "rom_range": float(val.abduccion_hombro.max() - val.abduccion_hombro.min()),
        "rom_p95": float(np.percentile(val.abduccion_hombro, 95)),
        # --- técnica del codo
        "codo_medio": float(val.flexion_codo.mean()),
        "codo_min": float(val.flexion_codo.min()),
        "codo_std": float(val.flexion_codo.std()),
        # --- compensaciones
        "tronco_max": float(val.inclinacion_tronco.max()),
        "tronco_medio": float(val.inclinacion_tronco.mean()),
        "lean_max": float(val.lean_lateral.max()),
        "lean_rango": float(val.lean_lateral.max() - val.lean_lateral.min()),
        "elevacion_escapular_max": float(val.elevacion_escapular.max()),
        "elevacion_escapular_rango": float(
            val.elevacion_escapular.max() - val.elevacion_escapular.min()),
        # --- simetría
        "contralateral_max": float(val.abduccion_contralateral.max()),
        "contralateral_medio": float(val.abduccion_contralateral.mean()),
        # --- control motor
        "duracion_s": float(rep["duracion_s"]),
        "ratio_con_exc": float(t_con / t_exc),
        "vel_pico": float(np.nanmax(np.abs(vel))),
        "vel_media": float(np.nanmean(np.abs(vel))),
        "suavidad_ldlj": ldlj(vel, dt),
        "tiempo_en_pico": float((val.abduccion_hombro >= umbral_pico).mean()),
        "variabilidad_ang": float(val.abduccion_hombro.std()),
        # --- contexto y calidad
        "visibilidad_media": float(val.visibilidad.mean()),
        "frac_valida": float(seg.valido.mean()),
        "largo_torso": float(val.largo_torso.mean()),
        "idx_rep": idx,
        "n_reps_video": n_reps,
        # progreso_serie queda deliberadamente fuera: valdría idx/(n_reps-1) y
        # exige conocer el total de repeticiones, dato inexistente en vivo hasta
        # que el usuario termina. Ver cont/PLAN.md, sección 9.1.
    }
print(f"geometria y variables listas: {len(COLUMNAS)} columnas de landmarks")

13 landmarks -> 93 columnas por CSV
umbrales: {'MIN_VISIBILIDAD': 0.5, 'SUAVIZADO_SEG': 0.4, 'MIN_DUR_REP': 1.0, 'MIN_AMPLITUD_REP': 20.0, 'MAX_DUR_REP': 20.0, 'MAX_LADO': 960.0}
Convierte un DataFrame de landmarks en las series angulares de un lado.
suavizado y segmentacion definidos en el propio notebook
geometria y variables listas: 93 columnas de landmarks


In [6]:
# ---------------------------------------------------------------------------
# Segmentacion causal: decidir sin ver el futuro
# ---------------------------------------------------------------------------
from collections import deque
from dataclasses import dataclass, field

class BufferCausal:
    """Savitzky-Golay centrado aplicado con retardo fijo.

    Cada muestra se emite cuando ya se dispone de `ventana // 2` muestras
    posteriores, que es exactamente lo que el filtro centrado necesita. El valor
    resultante coincide con el que produciría `savgol_filter` sobre la señal
    completa en los puntos interiores.

    Args:
        fps: tasa de frames, para derivar la ventana.
        orden: orden del polinomio, 3 como en el entrenamiento.
    """

    def __init__(self, fps: float, orden: int = 3) -> None:
        self.ventana = ventana_savgol(fps)
        if self.ventana <= orden:
            self.ventana = orden + 2 | 1
        self.orden = orden
        self.retardo = self.ventana // 2
        self._crudo: deque[float] = deque(maxlen=self.ventana)
        self._n_vistas = 0

    def append(self, valor: float) -> tuple[int, float] | None:
        """Añade una muestra cruda.

        Returns:
            ``(indice, valor_suavizado)`` de la muestra que queda definitiva con
            esta llegada, o ``None`` si aún no hay ventana completa.
        """
        self._crudo.append(float(valor))
        self._n_vistas += 1
        if len(self._crudo) < self.ventana:
            return None
        ventana = np.asarray(self._crudo, dtype=float)
        if not np.all(np.isfinite(ventana)):
            ventana = pd.Series(ventana).interpolate(
                limit_direction="both").to_numpy()
        if not np.all(np.isfinite(ventana)):
            return None
        suave = savgol_filter(ventana, self.ventana, self.orden)
        idx = self._n_vistas - 1 - self.retardo
        return idx, float(suave[self.retardo])

@dataclass
class _Extremo:
    idx: int
    valor: float

class SegmentadorOnline:
    """Detecta repeticiones valle → pico → valle de forma causal.

    Args:
        fps: tasa de frames.
        prominencia: salto mínimo, en grados, para confirmar un extremo. Si es
            ``None`` se calibra durante el calentamiento a partir del recorrido
            observado, con `prominencia_para` de la celda anterior.
    """

    REPOSO = "reposo"
    SUBIENDO = "subiendo"
    BAJANDO = "bajando"

    def __init__(self, fps: float, prominencia: float | None = None) -> None:
        self.fps = fps
        self.prominencia = prominencia
        self.estado = self.REPOSO
        self._valle_previo: _Extremo | None = None
        self._min: _Extremo | None = None
        self._max: _Extremo | None = None
        self._calibracion: list[float] = []
        self.repeticiones: list[dict] = []

    # -- calibración --------------------------------------------------------- #

    def _calibrar(self, valor: float) -> bool:
        """Acumula muestras hasta poder fijar la prominencia. True si ya está."""
        if self.prominencia is not None:
            return True
        self._calibracion.append(valor)
        # Con menos de 3 s no hay recorrido fiable del que derivar el umbral.
        if len(self._calibracion) < int(3 * self.fps):
            return False
        self.prominencia = prominencia_para(np.asarray(self._calibracion))
        return True

    # -- actualización ------------------------------------------------------- #

    def update(self, idx: int, valor: float) -> dict | None:
        """Procesa una muestra suavizada.

        Returns:
            El dict de la repetición si esta muestra la cierra, si no ``None``.
        """
        if not np.isfinite(valor) or not self._calibrar(valor):
            return None

        p = self.prominencia
        punto = _Extremo(idx, valor)

        if self.estado == self.REPOSO:
            if self._min is None or valor < self._min.valor:
                self._min = punto
            if valor > self._min.valor + p:
                self._valle_previo = self._min
                self._max = punto
                self.estado = self.SUBIENDO
            return None

        if self.estado == self.SUBIENDO:
            if valor > self._max.valor:
                self._max = punto
            if valor < self._max.valor - p:
                self._min = punto
                self.estado = self.BAJANDO
            return None

        # BAJANDO: se busca el valle que cierra la repetición.
        if valor < self._min.valor:
            self._min = punto
        if valor > self._min.valor + p * 0.6:
            rep = self._cerrar(self._valle_previo, self._max, self._min)
            self._valle_previo = self._min
            self._max = punto
            self.estado = self.SUBIENDO
            return rep
        return None

    def _cerrar(self, valle_ini: _Extremo, pico: _Extremo,
                valle_fin: _Extremo) -> dict | None:
        """Aplica los mismos filtros de amplitud y duración que la ruta por lotes."""
        amplitud = pico.valor - max(valle_ini.valor, valle_fin.valor)
        duracion = (valle_fin.idx - valle_ini.idx) / self.fps
        if (amplitud < MIN_AMPLITUD_REP or duracion < MIN_DUR_REP
                or duracion > MAX_DUR_REP):
            return None
        rep = {"inicio": valle_ini.idx, "pico": pico.idx, "fin": valle_fin.idx,
               "amplitud": float(amplitud), "duracion_s": float(duracion)}
        self.repeticiones.append(rep)
        return rep

In [7]:
# ---------------------------------------------------------------------------
# Acumulador de sesion: de fotogramas sueltos a repeticiones clasificables
# ---------------------------------------------------------------------------
@dataclass
class MetricasInstantaneas:
    """Realimentación de nivel A: se muestra en cada frame, sin modelo."""
    abduccion: float
    inclinacion_tronco: float
    fase: str
    repeticiones: int
    lado: str | None
    calibrando: bool

@dataclass
class AcumuladorEnVivo:
    """Estado de una sesión de cámara. Una instancia por usuario.

    Recibe landmarks frame a frame y produce dos cosas:

    - `MetricasInstantaneas` en cada frame, para pintar en pantalla.
    - El dict de variables de una repetición cuando esta se cierra, listo para
      `clasificar()`.

    **Detección del lado activo.** El brazo que trabaja cambia entre sujetos, y
    en vivo no se puede mirar el video entero para decidirlo. Una ventana fija
    tampoco sirve, y no por la razón evidente: en este protocolo los sujetos
    mueven **los dos** brazos, el activo simplemente más. Medido sobre Ex1, a los
    3 s ambos lados presentan 75-92° de recorrido y la razón entre ellos es de
    1.03-1.07, indistinguible del ruido; con una ventana fija de 6 s se elige el
    brazo equivocado en 3 de cada 4 sujetos, y entonces no se detecta ni una
    repetición porque el brazo escogido apenas se mueve.

    Lo que discrimina es la **razón** entre recorridos, que tarda entre 5 y 12 s
    en superar `RAZON_LADO_MINIMA`. El calentamiento por tanto no termina por
    tiempo sino cuando hay un ganador claro, y las repeticiones ocurridas
    mientras tanto no se pierden: el búfer se reprocesa con el lado ya decidido.
    `calentamiento_max_s` es la red de seguridad para no esperar indefinidamente.
    """

    #: Cuánto debe superar el lado activo al pasivo para darlo por bueno.
    RAZON_LADO_MINIMA = 1.15
    #: Comprobaciones consecutivas que deben coincidir antes de comprometerse.
    CONFIRMACIONES_LADO = 3

    fps: float = FPS_NOMINAL
    calentamiento_s: float = 8.0        # mínimo antes de la primera comprobación
    calentamiento_max_s: float = 40.0   # tope: pasado esto se decide igualmente
    lado: str | None = None

    _filas_calentamiento: list[dict] = field(default_factory=list, init=False)
    _serie: deque = field(default_factory=lambda: deque(maxlen=4000), init=False)
    _pendientes: deque = field(default_factory=deque, init=False)
    _buffer: BufferCausal | None = field(default=None, init=False)
    _segmentador: SegmentadorOnline | None = field(default=None, init=False)
    _n_frames: int = field(default=0, init=False)
    _n_reps: int = field(default=0, init=False)
    _lado_candidato: str | None = field(default=None, init=False)
    _confirmaciones: int = field(default=0, init=False)

    def __post_init__(self) -> None:
        self._buffer = BufferCausal(self.fps)
        self._segmentador = SegmentadorOnline(self.fps)

    @property
    def calibrando(self) -> bool:
        return self.lado is None

    def update(self, fila_landmarks: dict) -> tuple[MetricasInstantaneas, dict | None]:
        """Procesa un frame.

        Args:
            fila_landmarks: un dict con las claves de `COLUMNAS` para ese frame.

        Returns:
            ``(metricas_instantaneas, variables_de_repeticion_o_None)``.
        """
        fila = dict(fila_landmarks)
        fila.setdefault("frame", self._n_frames)
        fila.setdefault("t_seg", self._n_frames / self.fps)
        self._n_frames += 1

        if self.lado is None:
            self._filas_calentamiento.append(fila)
            if self._intentar_decidir_lado():
                self._recalibrar_prominencia()
                # Reproducir el calentamiento con el lado ya decidido, para no
                # perder las repeticiones ocurridas durante él.
                for f in self._filas_calentamiento:
                    _, variables = self._procesar(f)
                    if variables:
                        self._pendientes.append(variables)
                self._filas_calentamiento.clear()
                return self._metricas(np.nan, np.nan), self._siguiente_pendiente()
            return self._metricas(np.nan, np.nan), None

        metricas, variables = self._procesar(fila)
        if variables:
            self._pendientes.append(variables)
        return metricas, self._siguiente_pendiente()

    def _siguiente_pendiente(self) -> dict | None:
        return self._pendientes.popleft() if self._pendientes else None

    def _recalibrar_prominencia(self) -> None:
        """Fija la prominencia con todo el calentamiento, ya con el lado decidido.

        El segmentador se autocalibra con sus primeras muestras, que pueden caer
        en un tramo de reposo y dar un umbral poco representativo. Una vez se
        sabe qué brazo trabaja, el búfer de calentamiento es una estimación
        mucho mejor del recorrido real.
        """
        if not self._filas_calentamiento:
            return
        df = pd.DataFrame(self._filas_calentamiento)
        s = series_angulares(df, self.lado)
        suave = suavizar(s["abduccion_hombro"].where(s.valido), self.fps)
        self._segmentador = SegmentadorOnline(
            self.fps, prominencia=prominencia_para(suave))

    def _intentar_decidir_lado(self) -> bool:
        """Fija `self.lado` si ya hay un ganador claro. True si se decidió.

        Precisión medida sobre las 13 vistas frontales de Ex1 (que es lo que ve
        una webcam): **11 de 13**. Los dos fallos son sujetos que mueven ambos
        brazos con recorridos casi iguales. Por eso la interfaz permite fijar el
        brazo a mano: el paciente sabe cuál está ejercitando.
        """
        n = len(self._filas_calentamiento)
        minimo = int(self.calentamiento_s * self.fps)
        if n < minimo:
            return False
        # Comprobar una vez por segundo, no en cada frame: detectar_lado
        # reconstruye las series completas y no es gratis.
        if n % int(self.fps) != 0 and n < int(self.calentamiento_max_s * self.fps):
            return False

        df = pd.DataFrame(self._filas_calentamiento)
        lado, rango_izq, rango_der = detectar_lado(df)
        mayor, menor = max(rango_izq, rango_der), min(rango_izq, rango_der)

        hay_movimiento = mayor >= MIN_AMPLITUD_REP
        hay_ganador = mayor >= self.RAZON_LADO_MINIMA * max(menor, 1e-6)
        agotado = n >= int(self.calentamiento_max_s * self.fps)

        # Exigir que varias comprobaciones seguidas coincidan. Una sola medición
        # favorable puede ser ruido: en los primeros segundos la razón entre
        # lados oscila y comprometerse con la primera lectura buena falla en la
        # mitad de los sujetos.
        if hay_movimiento and hay_ganador and lado == self._lado_candidato:
            self._confirmaciones += 1
        else:
            self._lado_candidato = lado if (hay_movimiento and hay_ganador) else None
            self._confirmaciones = 1 if self._lado_candidato else 0

        if agotado or self._confirmaciones >= self.CONFIRMACIONES_LADO:
            self.lado = lado
            return True
        return False

    def _procesar(self, fila: dict) -> tuple[MetricasInstantaneas, dict | None]:
        s = series_angulares(pd.DataFrame([fila]), self.lado)
        self._serie.append(s.iloc[0].to_dict())

        abduccion = float(s.iloc[0]["abduccion_hombro"])
        tronco = float(s.iloc[0]["inclinacion_tronco"])

        emitido = self._buffer.append(abduccion)
        rep_features = None
        if emitido is not None:
            idx, suave = emitido
            self._anotar_suave(idx, suave)
            rep = self._segmentador.update(idx, suave)
            if rep is not None:
                rep_features = self._variables(rep)
        return self._metricas(abduccion, tronco), rep_features

    def _anotar_suave(self, idx: int, valor: float) -> None:
        pos = idx - (self._n_frames - len(self._serie))
        if 0 <= pos < len(self._serie):
            self._serie[pos]["abduccion_suave"] = valor

    def _variables(self, rep: dict) -> dict | None:
        """Construye el dict de variables reutilizando el código del entrenamiento."""
        base = self._n_frames - len(self._serie)
        ini, fin = rep["inicio"] - base, rep["fin"] - base
        if ini < 0:
            return None   # la repetición se salió del búfer
        df = pd.DataFrame(list(self._serie)[ini:fin + 1])
        if "abduccion_suave" not in df or df["abduccion_suave"].isna().all():
            return None
        df["abduccion_suave"] = df["abduccion_suave"].interpolate(limit_direction="both")
        rep_local = {**rep, "inicio": 0, "pico": rep["pico"] - rep["inicio"],
                     "fin": fin - ini}
        variables = features_repeticion(df, rep_local, self.fps,
                                           self._n_reps, self._n_reps + 1)
        if variables:
            self._n_reps += 1
            variables["es_lateral"] = 0   # la webcam siempre es vista frontal
        return variables or None

    def _metricas(self, abduccion: float, tronco: float) -> MetricasInstantaneas:
        return MetricasInstantaneas(
            abduccion=abduccion,
            inclinacion_tronco=tronco,
            fase=self._segmentador.estado if self.lado else "calibrando",
            repeticiones=self._n_reps,
            lado=self.lado,
            calibrando=self.calibrando,
        )

print("segmentador causal y acumulador definidos")

segmentador causal y acumulador definidos


In [8]:
# ---------------------------------------------------------------------------
# Pesos de MediaPipe Pose (API Tasks)
#
# mediapipe 0.10.35 ya no trae la API legacy `mp.solutions`: `PoseLandmarker`
# necesita un archivo `.task`, que se descarga una sola vez a models/mediapipe/.
#
# La variante forma parte del contrato del modelo. El entrenamiento uso "heavy"
# porque era un proceso por lotes y primaba la precision; aqui hay una persona
# esperando delante de la camara, asi que se usa "full". No es gratis: medido
# sobre PM_000, la abduccion maxima pasa de 147 a 168 grados entre una y otra.
# ---------------------------------------------------------------------------
import urllib.request

import mediapipe as mp

URLS_MEDIAPIPE = {
    "lite": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
            "pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    "full": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
            "pose_landmarker_full/float16/1/pose_landmarker_full.task",
    "heavy": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
             "pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task",
}


def descargar_modelo(variante: str = "full") -> Path:
    """Descarga el `.task` una sola vez. Idempotente."""
    destino = MODEL_DIR / "mediapipe" / f"pose_landmarker_{variante}.task"
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists() or destino.stat().st_size < 1_000_000:
        print(f"Descargando pesos {variante} (una sola vez)...")
        urllib.request.urlretrieve(URLS_MEDIAPIPE[variante], destino)
    return destino


MP_MODEL = descargar_modelo(VARIANTE_MP)
print(f"Modelo MediaPipe: {MP_MODEL.name}  ({MP_MODEL.stat().st_size/1e6:.1f} MB)")
print(f"mediapipe {mp.__version__} — API Tasks, reescalado a {MAX_LADO} px")


Modelo MediaPipe: pose_landmarker_full.task  (9.4 MB)
mediapipe 0.10.35 — API Tasks, reescalado a 960 px


In [9]:
# ---------------------------------------------------------------------------
# Detector fotograma a fotograma
#
# El de entrenamiento recorria un video entero; este recibe imagenes sueltas,
# que es lo que llega de una camara. Se usa el modo VIDEO de MediaPipe y no
# LIVE_STREAM: el modo en vivo es asincrono y devuelve los resultados por
# callback, lo que reordena los fotogramas y rompe la segmentacion.
#
# Dos cuidados que el de entrenamiento no necesitaba, porque alli habia un solo
# hilo recorriendo un archivo:
#
# 1. La marca de tiempo sale del **reloj**, no de un contador de fotogramas. Una
#    camara no entrega 30 por segundo por mucho que se le pidan, y contar como
#    si lo hiciera le miente al seguimiento de MediaPipe.
# 2. El landmarker en modo VIDEO exige marcas estrictamente crecientes y no es
#    seguro entre hilos: repetir una marca lo aborta con "Input timestamp must
#    be monotonically increasing", y una excepcion dentro del manejador mata el
#    stream de Gradio. Con `concurrency_limit > 1` dos fotogramas pueden entrar
#    a la vez, asi que la inferencia se serializa con un cerrojo.
# ---------------------------------------------------------------------------
import threading
import time

class DetectorPorFotograma:
    """Envuelve el PoseLandmarker para procesar imagen a imagen."""

    def __init__(self, variante: str = VARIANTE_MP,
                 max_lado: int = MAX_LADO) -> None:
        self.max_lado = max_lado
        opciones = mp.tasks.vision.PoseLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(
                model_asset_path=str(descargar_modelo(variante))),
            running_mode=mp.tasks.vision.RunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        self._detector = mp.tasks.vision.PoseLandmarker.create_from_options(opciones)
        self._vacias = {c: np.nan for c in COLUMNAS if c not in ("frame", "t_seg")}
        self._lock = threading.Lock()
        self._ultimo_ms = -1
        self._t0 = time.monotonic()

    def _marca(self, ms: int | None) -> int:
        """Marca de tiempo en milisegundos, estrictamente creciente.

        `None` significa "usa el reloj". El `+1` cubre el empate: dos fotogramas
        en el mismo milisegundo no son un error, pero repetir la marca si lo es.
        """
        if ms is None:
            ms = int((time.monotonic() - self._t0) * 1000)
        if ms <= self._ultimo_ms:
            ms = self._ultimo_ms + 1
        self._ultimo_ms = ms
        return ms

    def procesar(self, frame_rgb: np.ndarray,
                 ms: int | None = None) -> tuple[dict, np.ndarray]:
        """Devuelve (fila de landmarks, imagen anotada)."""
        alto, ancho = frame_rgb.shape[:2]
        escala = self.max_lado / max(alto, ancho)
        imagen_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
        if escala < 1.0:
            reducida = cv2.resize(imagen_bgr, (int(ancho * escala), int(alto * escala)),
                                  interpolation=cv2.INTER_AREA)
        else:
            reducida = imagen_bgr

        imagen = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(reducida, cv2.COLOR_BGR2RGB))
        # El reescalado de arriba queda fuera del cerrojo a proposito: lo unico
        # que hay que serializar es la inferencia.
        with self._lock:
            resultado = self._detector.detect_for_video(imagen, self._marca(ms))

        fila = dict(self._vacias)
        anotado = frame_rgb
        if resultado.pose_landmarks:
            normales, mundo = resultado.pose_landmarks[0], resultado.pose_world_landmarks[0]
            for indice_lm, nombre in zip(LANDMARKS, NOMBRES):
                p, pw = normales[indice_lm], mundo[indice_lm]
                fila.update({
                    f"{nombre}_x": p.x, f"{nombre}_y": p.y, f"{nombre}_z": p.z,
                    f"{nombre}_v": p.visibility,
                    f"{nombre}_wx": pw.x, f"{nombre}_wy": pw.y, f"{nombre}_wz": pw.z,
                })
            anotado = dibujar_esqueleto(frame_rgb.copy(), normales)
        return fila, anotado

    def cerrar(self) -> None:
        self._detector.close()


#: Conexiones que se dibujan sobre la imagen. Solo las del tronco y los brazos:
#: son las que intervienen en el gesto, y pintar el esqueleto entero distrae.
CONEXIONES = [(11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
              (11, 23), (12, 24), (23, 24)]


def dibujar_esqueleto(frame_rgb: np.ndarray, landmarks) -> np.ndarray:
    alto, ancho = frame_rgb.shape[:2]
    puntos = {i: (int(landmarks[i].x * ancho), int(landmarks[i].y * alto))
              for i in {p for conexion in CONEXIONES for p in conexion}}
    for a, b in CONEXIONES:
        cv2.line(frame_rgb, puntos[a], puntos[b], (34, 197, 94), 3)
    for punto in puntos.values():
        cv2.circle(frame_rgb, punto, 5, (34, 197, 94), -1)
    return frame_rgb


def para_pantalla(frame_rgb: np.ndarray | None) -> np.ndarray | None:
    """Reduce la imagen anotada antes de devolverla al navegador.

    Solo toca lo que se ve. Devolver el fotograma al tamano de entrada manda
    mas pixeles de los que caben en la zona de video, y cada uno de ellos vuelve
    por la red 30 veces por segundo.
    """
    if frame_rgb is None or ANCHO_SALIDA <= 0 or frame_rgb.shape[1] <= ANCHO_SALIDA:
        return frame_rgb
    alto = round(frame_rgb.shape[0] * ANCHO_SALIDA / frame_rgb.shape[1])
    return cv2.resize(frame_rgb, (ANCHO_SALIDA, alto), interpolation=cv2.INTER_AREA)


print("detector por fotograma definido")

detector por fotograma definido


In [10]:
# ---------------------------------------------------------------------------
# Sesion en vivo: lo que hace la aplicacion, en una clase
#
# Recibe fotogramas y devuelve, en cada uno, las metricas del instante; y solo
# cuando una repeticion se cierra, su clasificacion y su consigna. Es el mismo
# reparto que en pantalla: los angulos se mueven todo el rato, el veredicto
# aparece una vez por repeticion.
# ---------------------------------------------------------------------------
import time
import unicodedata

#: Cuanto permanece el aviso de la ultima repeticion sobre la imagen. Lo
#: bastante para leerlo, no tanto como para tapar la repeticion siguiente.
DURACION_AVISO_S = 3.0

#: Color del aviso por clase, en RGB —los fotogramas de Gradio llegan en RGB—.
COLOR_POR_CLASE = {
    "correcto": (79, 157, 105),
    "rango_insuficiente": (217, 119, 87),
    "compensacion_tronco": (107, 127, 215),
}


def _a_ascii(texto: str) -> str:
    """Translitera a ASCII para poder pintarlo con `cv2.putText`.

    OpenCV solo dibuja ASCII: una tilde o un signo de apertura salen como
    interrogante, y las consignas llevan ambos ("¡Bien hecho!", "Manten el
    torso recto").
    """
    sin_tildes = "".join(c for c in unicodedata.normalize("NFKD", texto)
                         if not unicodedata.combining(c))
    return sin_tildes.encode("ascii", "ignore").decode("ascii").strip()


class SesionNotebook:
    """Version reducida de la sesion en vivo de la aplicacion."""

    def __init__(self, fps: float = 30.0, lado: str | None = None) -> None:
        self.fps = fps
        self.detector = DetectorPorFotograma()
        self.acumulador = AcumuladorEnVivo(fps=fps, lado=lado)
        self.repeticiones: list[dict] = []
        self._instantes: deque = deque(maxlen=30)
        self._aviso: tuple[dict, float] | None = None

    @property
    def fps_real(self) -> float:
        """Fotogramas por segundo que estan llegando de verdad.

        Conviene tenerlo a la vista: `fps` es la tasa que supone la
        segmentacion, y si la camara entrega la mitad las duraciones y las
        velocidades de cada repeticion se miden mal aunque nada falle.
        """
        if len(self._instantes) < 2:
            return 0.0
        lapso = self._instantes[-1] - self._instantes[0]
        return (len(self._instantes) - 1) / lapso if lapso > 0 else 0.0

    def procesar(self, frame_rgb: np.ndarray):
        """Devuelve (imagen anotada, metricas, resultado_o_None)."""
        self._instantes.append(time.monotonic())

        # Sin marca de tiempo: el detector usa el reloj, que es lo que
        # corresponde cuando los fotogramas llegan cuando pueden.
        fila, anotado = self.detector.procesar(frame_rgb)
        if np.isnan(fila.get("left_shoulder_wx", np.nan)):
            return self._con_aviso(anotado), None, None   # sin persona en el encuadre

        metricas, variables = self.acumulador.update(fila)
        if not variables:
            return self._con_aviso(anotado), metricas, None

        # La repeticion acaba de cerrarse: aqui es donde entra el modelo.
        resultado = clasificar(variables)
        resultado["indice"] = len(self.repeticiones) + 1
        resultado["variables"] = variables
        resultado["consigna"] = consigna(resultado["label"],
                                         indice=len(self.repeticiones))
        self.repeticiones.append(resultado)
        self._aviso = (resultado, time.monotonic())
        return self._con_aviso(anotado), metricas, resultado

    def _con_aviso(self, frame_rgb: np.ndarray) -> np.ndarray:
        """Superpone el veredicto de la ultima repeticion sobre la imagen.

        El panel lateral ya lo dice, pero quien esta ejercitando se mira a si
        mismo, no al panel: la correccion tiene que estar donde tiene los ojos.
        El aviso se borra a los `DURACION_AVISO_S` segundos para no tapar la
        repeticion siguiente.
        """
        if self._aviso is None:
            return frame_rgb
        resultado, instante = self._aviso
        if time.monotonic() - instante > DURACION_AVISO_S:
            self._aviso = None
            return frame_rgb

        color = COLOR_POR_CLASE.get(resultado["label"], (128, 128, 128))
        etiqueta = (f"REP {resultado['indice']}: "
                    f"{_a_ascii(resultado['consigna']).upper()}")

        salida = frame_rgb.copy()
        alto, ancho = salida.shape[:2]
        escala = max(0.5, ancho / 1100)
        grosor = max(1, int(ancho / 640))
        (ancho_txt, alto_txt), _ = cv2.getTextSize(
            etiqueta, cv2.FONT_HERSHEY_SIMPLEX, escala, grosor)

        margen = int(alto_txt * 0.6)
        x0, y0 = margen, margen
        x1 = min(ancho - margen, x0 + ancho_txt + 2 * margen)
        y1 = y0 + alto_txt + 2 * margen

        # Banda semitransparente: el texto tiene que leerse sobre cualquier fondo.
        capa = salida.copy()
        cv2.rectangle(capa, (x0, y0), (x1, y1), color, -1)
        cv2.addWeighted(capa, 0.75, salida, 0.25, 0, salida)
        cv2.putText(salida, etiqueta, (x0 + margen, y1 - margen),
                    cv2.FONT_HERSHEY_SIMPLEX, escala, (255, 255, 255), grosor,
                    cv2.LINE_AA)
        return salida

    def resumen(self) -> dict:
        etiquetas = [r["label"] for r in self.repeticiones]
        errores = [e for e in etiquetas if e != "correcto"]
        dominante = (max(set(errores), key=errores.count)
                     if len(errores) >= max(1, len(etiquetas) / 3) else "correcto")
        roms = [r["variables"].get("rom_max", 0.0) for r in self.repeticiones]
        return {"repeticiones": len(etiquetas),
                "correctas": sum(1 for e in etiquetas if e == "correcto"),
                "clasificacion": dominante,
                "por_repeticion": etiquetas,
                "rom_max": max(roms) if roms else 0.0,
                "rom_medio": float(np.mean(roms)) if roms else 0.0,
                "lado": self.acumulador.lado}

    def cerrar(self) -> None:
        self.detector.cerrar()


print("sesion en vivo definida")


sesion en vivo definida


### 4.1 Una sesion con la camara, repeticion a repeticion

La camara del navegador entra a 30 Hz y cada fotograma recorre el camino
completo: pose, angulos, bufer causal, segmentador. Cuando el segmentador cierra
una repeticion —y solo entonces— el modelo la clasifica y su veredicto aparece
**sobre la imagen y en el registro, en ese mismo instante**.

Es la realimentacion que da la app PhysioVision: el paciente se esta mirando a si
mismo, no al panel, asi que la correccion se pinta encima del video y dura unos
segundos, los justos para leerla antes de la repeticion siguiente.

Aqui la ruta va **desnuda**: sin Gemini y sin voz, que son las secciones 5 y 6, y
sin historial ni sesiones. Lo que se comprueba es lo unico que no se puede dar
por hecho — que el modelo reacciona repeticion a repeticion, en tiempo real, sin
ver el futuro. La seccion 7 vuelve a montar lo mismo ya con la redaccion y el
audio.

En Colab la camara se abre con un enlace publico temporal (`share=True`): el
servidor esta en la nube, pero los fotogramas salen de tu navegador.


In [11]:
# Requiere el modelo entrenado (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_MODELO:
    print("Se salta: falta el modelo entrenado.")
else:
    import gradio as gr

    def iniciar_prueba(brazo: str):
        """Crea la sesion. Vive en el estado del navegador, no en una global:
        el detector de MediaPipe tiene estado y no es seguro entre hilos."""
        lado = brazo if brazo in ("left", "right") else None
        return (SesionNotebook(fps=30.0, lado=lado),
                "Sesion iniciada. Colocate de cuerpo entero frente a la camara.",
                "")

    def procesar_prueba(frame, sesion, registro):
        """Se ejecuta en cada fotograma que llega. Escribe en el registro solo
        cuando una repeticion se cierra: el resto del tiempo devuelve
        `gr.skip()`."""
        if sesion is None or frame is None:
            return frame, gr.skip(), gr.skip(), sesion

        anotado, metricas, resultado = sesion.procesar(frame)
        pantalla = para_pantalla(anotado)

        if metricas is None:
            return pantalla, "Sin persona en el encuadre", gr.skip(), sesion

        tasa = f" · {sesion.fps_real:.0f} fps"
        if metricas.calibrando:
            estado = "Calibrando: detectando que brazo trabaja..." + tasa
        else:
            estado = (f"**{metricas.fase}** · hombro {metricas.abduccion:.0f} grados"
                      f" · tronco {metricas.inclinacion_tronco:.0f}"
                      f" · repeticiones {metricas.repeticiones}{tasa}")

        if resultado is None:
            return pantalla, estado, gr.skip(), sesion

        # Repeticion cerrada: el veredicto, en el instante en que ocurre.
        v = resultado["variables"]
        linea = (f"REP {resultado['indice']:2d}  {resultado['label']:22s}"
                 f" {resultado['confidence']:4.0%}   rango {v['rom_max']:3.0f}"
                 f"   tronco {v['tronco_max']:3.0f}   {v['duracion_s']:.1f} s\n"
                 f"        -> {resultado['consigna']}\n")
        return pantalla, estado, registro + linea, sesion

    def terminar_prueba(sesion, registro):
        """Cierra la serie. El resumen es un extra: lo que importa ya se dijo."""
        if sesion is None:
            return None, "No hay ninguna sesion activa.", registro
        r = sesion.resumen()
        sesion.cerrar()
        if r["repeticiones"] == 0:
            return None, "No se detecto ninguna repeticion completa.", registro
        cierre = (f"\n{r['correctas']}/{r['repeticiones']} correctas · "
                  f"predominante {r['clasificacion']} · rango maximo "
                  f"{r['rom_max']:.0f} grados · brazo {r['lado']}\n")
        return None, "Serie terminada.", registro + cierre

    with gr.Blocks(title="PhysioVision — ruta causal") as prueba:
        gr.Markdown(
            "### La ruta causal, en directo\n"
            "Cada linea del registro aparece **en el instante en que la repeticion "
            "se cierra**, no al final de la serie. Sin Gemini y sin voz: eso es la "
            "seccion 7.")

        sesion_prueba = gr.State(None)

        with gr.Row():
            brazo_prueba = gr.Dropdown(
                label="Brazo",
                choices=[("Detectar solo", "auto"), ("Derecho", "right"),
                         ("Izquierdo", "left")],
                value="right", scale=1,
                info="Indicarlo evita el calentamiento de deteccion del lado.")
            boton_iniciar_prueba = gr.Button("Iniciar", variant="primary", scale=0)
            boton_terminar_prueba = gr.Button("Terminar serie", scale=0)

        with gr.Row():
            with gr.Column(scale=3):
                camara_prueba = gr.Image(
                    label="Camara", sources=["webcam"], streaming=True,
                    type="numpy",
                    # Sin espejo: reflejar intercambiaria izquierda y derecha y
                    # el seguimiento acabaria en el brazo equivocado. Y con la
                    # resolucion acotada: ver RESOLUCION_CAMARA.
                    webcam_options=gr.WebcamOptions(
                        mirror=False, constraints=RESOLUCION_CAMARA))
                salida_prueba = gr.Image(label="Seguimiento", interactive=False)
            with gr.Column(scale=2):
                estado_prueba = gr.Markdown("Sin sesion activa")
                registro = gr.Textbox(label="Repeticiones", lines=16, max_lines=16,
                                      interactive=False, autoscroll=True)

        boton_iniciar_prueba.click(iniciar_prueba, [brazo_prueba],
                                   [sesion_prueba, estado_prueba, registro])
        boton_terminar_prueba.click(terminar_prueba, [sesion_prueba, registro],
                                    [sesion_prueba, estado_prueba, registro])
        camara_prueba.stream(
            procesar_prueba, [camara_prueba, sesion_prueba, registro],
            [salida_prueba, estado_prueba, registro, sesion_prueba],
            # 30 Hz es un techo, no una promesa: la tasa real la fija el
            # viaje de ida y vuelta de cada fotograma.
            stream_every=1 / 30, concurrency_limit=2, show_progress="hidden")

    prueba.queue(default_concurrency_limit=2).launch(
        inline=not EN_COLAB, share=EN_COLAB, quiet=True, height=760)


### 4.2 Parar el servidor de la prueba


In [12]:
# Para el servidor de la prueba. Conviene hacerlo antes de la seccion 7:
# dos servidores con la camara abierta se pelean por el dispositivo.
if "prueba" in dir():
    prueba.close()
    print("Prueba detenida.")


Closing server running on port: 7860
Prueba detenida.


---
## 5. Redaccion de la realimentacion con Google Gemini

El modelo dice *que* falla; `knowledge_base/ejercicios.json` dice *que hacer*.
Ninguno de los dos suena a persona: el texto del JSON es identico para todas las
repeticiones de la misma clase, diga 95 grados de rango o 140.

Gemini se ocupa solo de eso: **reformular** la recomendacion ya validada usando
las metricas concretas de esa repeticion. No clasifica, no diagnostica y no
inventa consejo clinico. La division de trabajo es:

| Pieza | Decide |
|---|---|
| XGBoost | la clase de la repeticion |
| `ejercicios.json` | que recomendar para esa clase |
| Gemini | como decirlo |

> **Por que importa la restriccion.** Un modelo de lenguaje generando consejo de
> rehabilitacion por su cuenta produce texto plausible y sin respaldo, y aqui lo
> lee alguien moviendose con un hombro lesionado. Por eso la instruccion de
> sistema se lo prohibe explicitamente y la advertencia de seguridad se copia
> literal del JSON, sin pasar por el modelo.

Esta seccion sirve para **afinar los prompts** contra repeticiones reales del
dataset antes de que lleguen a un paciente. Estan copiados palabra por palabra de
los que usa la app en la celda siguiente: lo que ajustes aqui es lo que hay que
llevar a produccion.

In [13]:
# ---------------------------------------------------------------------------
# Redactor con Gemini, en su version minima
#
# La de la aplicacion anade reintentos, cortacircuito
# y tiempos de espera, que aqui solo estorbarian. Lo que SI se conserva palabra
# por palabra es la instruccion de sistema y los prompts: son la barrera que
# impide que el modelo se salga del guion, y cambiarlos cambia lo que se le dice
# al paciente.
# ---------------------------------------------------------------------------
import os

INSTRUCCION_SISTEMA = """\
Eres el asistente de redacción de PhysioVision, una app de apoyo a ejercicios de
rehabilitación de hombro. Tu único trabajo es reescribir una recomendación ya
validada para que suene cercana y concreta.

REGLAS INVIOLABLES:
1. No inventes consejo clínico. Reformula SOLO lo que te den en RECOMENDACION.
2. No diagnostiques, no menciones patologías, no sugieras ejercicios nuevos, no
   propongas cambiar series, repeticiones ni cargas.
3. Puedes citar las métricas que te den para hacer el mensaje concreto.
4. Dirígete al paciente de tú, en español, con tono cálido y sereno.
5. Nunca alarmes. Si el resultado es un error de técnica, enmárcalo como un
   ajuste, no como un fallo.
6. Sin markdown, sin listas, sin emojis, sin comillas. Texto corrido.
7. Si los datos son contradictorios o insuficientes, limítate a reformular la
   recomendación sin citar números.
"""

PROMPT_REPETICION = """\
Repetición {indice} de la serie.

RESULTADO: {etiqueta}
CONTEXTO (para elegir el matiz, NO para citarlo):
- Rango alcanzado: {rom_max:.0f} grados
- Inclinación del tronco: {tronco_max:.0f} grados
- Duración: {duracion_s:.1f} segundos
- Brazo: {lado}

CONSIGNA (reformula esto, no la sustituyas):
{recomendacion}

Devuelve UNA consigna de 2 a 6 palabras, en imperativo, para que la oiga entre
esta repetición y la siguiente.

Si el RESULTADO es "correcto", devuelve solo un elogio breve: "¡Correcto!",
"¡Bien hecho!", "¡Así es!" o similar. Nada más.

Si no lo es, di qué corregir en el gesto: "Sube más el brazo", "Mantén el torso
recto", "Hazlo más lento". Una sola indicación, la más importante.\
"""

PROMPT_RESUMEN = """\
Fin de la serie.

REPETICIONES: {total} en total, {correctas} correctas
RESULTADO PREDOMINANTE: {etiqueta}
SECUENCIA: {secuencia}
RANGO DE MOVIMIENTO: máximo {rom_max:.0f} grados, medio {rom_medio:.0f} grados
BRAZO: {lado}

RECOMENDACION (reformula esto, no la sustituyas):
{recomendacion}

Escribe dos o tres frases (50-80 palabras): primero cómo ha ido la serie en
conjunto, luego en qué concentrarse la próxima vez. Si la secuencia muestra que
los errores se acumulan al final, puedes mencionar la fatiga.\
"""


MODELO_GEMINI = os.environ.get("PHYSIOVISION_GEMINI_MODELO",
                               "gemini-flash-lite-latest")


class RedactorMinimo:
    """Reformula una recomendacion ya validada. Nunca decide que recomendar."""

    def __init__(self) -> None:
        self.cliente = None
        self.motivo = "sin GEMINI_API_KEY"
        clave = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
        if not clave:
            return
        try:
            from google import genai
            self.genai = genai
            self.cliente = genai.Client(api_key=clave)
            self.motivo = ""
        except Exception as error:            # sin biblioteca o clave invalida
            self.motivo = f"{type(error).__name__}: {error}"

    @property
    def disponible(self) -> bool:
        return self.cliente is not None

    def _generar(self, prompt: str) -> str | None:
        if not self.disponible:
            return None
        try:
            respuesta = self.cliente.models.generate_content(
                model=MODELO_GEMINI,
                contents=prompt,
                config=self.genai.types.GenerateContentConfig(
                    system_instruction=INSTRUCCION_SISTEMA,
                    temperature=0.7, max_output_tokens=2048),
            )
            return (respuesta.text or "").strip() or None
        except Exception as error:
            print(f"  (Gemini no respondio: {type(error).__name__})")
            return None

    def repeticion(self, indice, etiqueta, confianza, recomendacion, variables, lado):
        return self._generar(PROMPT_REPETICION.format(
            indice=indice, etiqueta=etiqueta, confianza=confianza,
            rom_max=variables.get("rom_max", float("nan")),
            tronco_max=variables.get("tronco_max", float("nan")),
            duracion_s=variables.get("duracion_s", float("nan")),
            lado=_lado_legible(lado), recomendacion=recomendacion))

    def resumen(self, resumen: dict, recomendacion: str):
        return self._generar(PROMPT_RESUMEN.format(
            total=resumen.get("repeticiones", 0),
            correctas=resumen.get("correctas", 0),
            etiqueta=resumen.get("clasificacion", "sin_datos"),
            secuencia=" -> ".join(resumen.get("por_repeticion", [])) or "sin datos",
            rom_max=resumen.get("rom_max", 0.0),
            rom_medio=resumen.get("rom_medio", 0.0),
            lado=_lado_legible(resumen.get("lado")),
            recomendacion=recomendacion))


def _lado_legible(lado: str | None) -> str:
    return {"left": "izquierdo", "right": "derecho"}.get(lado or "", "no determinado")


redactor = RedactorMinimo()
print("Gemini disponible —", MODELO_GEMINI if redactor.disponible
      else f"NO: {redactor.motivo}")


Gemini disponible — gemini-flash-lite-latest


In [14]:
# La instruccion de sistema, que es donde viven las restricciones de seguridad.
print(INSTRUCCION_SISTEMA)

Eres el asistente de redacción de PhysioVision, una app de apoyo a ejercicios de
rehabilitación de hombro. Tu único trabajo es reescribir una recomendación ya
validada para que suene cercana y concreta.

REGLAS INVIOLABLES:
1. No inventes consejo clínico. Reformula SOLO lo que te den en RECOMENDACION.
2. No diagnostiques, no menciones patologías, no sugieras ejercicios nuevos, no
   propongas cambiar series, repeticiones ni cargas.
3. Puedes citar las métricas que te den para hacer el mensaje concreto.
4. Dirígete al paciente de tú, en español, con tono cálido y sereno.
5. Nunca alarmes. Si el resultado es un error de técnica, enmárcalo como un
   ajuste, no como un fallo.
6. Sin markdown, sin listas, sin emojis, sin comillas. Texto corrido.
7. Si los datos son contradictorios o insuficientes, limítate a reformular la
   recomendación sin citar números.



### 5.1 Comparacion sobre repeticiones reales

Se toma una repeticion de cada clase, de las que el modelo acaba de clasificar, y
se enfrenta el texto del JSON con el redactado. Es la forma de ver si el prompt
produce algo util o solo adorno.

In [15]:
# Requiere modelo y dataset (seccion 0). Sin ellos se salta y el notebook sigue.
if not (HAY_DATASET and HAY_MODELO):
    print("Se salta: hacen falta el modelo y el dataset de repeticiones.")
else:
    # `CONOCIMIENTO`, `recomendacion()` y `consigna()` se definieron en la seccion 2.

    # Una repeticion representativa por clase.
    ejemplos = []
    for clase in sorted(df_reps.etiqueta.unique()):
        sub = df_reps[df_reps.etiqueta == clase]
        if not len(sub):
            continue
        fila = sub.iloc[len(sub) // 2]
        ejemplos.append((clase, fila))

    # Tras cada repeticion se muestra una CONSIGNA corta, no un parrafo: quien acaba
    # de moverse tiene un par de segundos antes de la siguiente. El texto largo se
    # reserva para el resumen del final de la serie.
    for clase, fila in ejemplos:
        base = consigna(clase, indice=int(fila.idx_rep))
        print("=" * 74)
        print(f"CLASE: {clase}   ({fila.sujeto}, repeticion {int(fila.idx_rep)})")
        print(f"  rom_max {fila.rom_max:.0f}  tronco_max {fila.tronco_max:.0f}  "
              f"duracion {fila.duracion_s:.1f}s")
        print("-" * 74)
        print(f"CONSIGNA BASE (json)   : {base}")
        if redactor.disponible:
            texto = redactor.repeticion(
                indice=int(fila.idx_rep) + 1, etiqueta=clase, confianza=0.9,
                recomendacion=base, variables=fila.to_dict(), lado=fila.lado)
            palabras = len(texto.split()) if texto else 0
            print(f"CONSIGNA REDACTADA (ia): {texto or '(sin respuesta)'}"
                  + (f"   [{palabras} palabras]" if texto else ""))
    print("=" * 74)
    print("\nSi alguna redaccion pasa de 6 palabras o deja de ser imperativa,")
    print("endurece PROMPT_REPETICION en la celda anterior.")


CLASE: compensacion_tronco   (PM_109, repeticion 1)
  rom_max 108  tronco_max 20  duracion 8.4s
--------------------------------------------------------------------------
CONSIGNA BASE (json)   : No inclines el cuerpo
CONSIGNA REDACTADA (ia): Mantén el torso recto   [4 palabras]
CLASE: correcto   (PM_032, repeticion 2)
  rom_max 18  tronco_max 7  duracion 2.9s
--------------------------------------------------------------------------
CONSIGNA BASE (json)   : ¡Así es!
CONSIGNA REDACTADA (ia): ¡Así es!   [2 palabras]
CLASE: rango_insuficiente   (PM_114, repeticion 19)
  rom_max 71  tronco_max 12  duracion 5.5s
--------------------------------------------------------------------------
CONSIGNA BASE (json)   : Eleva un poco más
CONSIGNA REDACTADA (ia): Eleva un poco más el brazo   [6 palabras]

Si alguna redaccion pasa de 6 palabras o deja de ser imperativa,
endurece PROMPT_REPETICION en la celda anterior.


### 5.2 Resumen de serie

El mensaje de cierre tiene mas contexto que el de una repeticion: la secuencia
completa. Con ella el modelo puede notar cosas que ninguna repeticion aislada
muestra, como que los errores se concentren al final por fatiga.

In [16]:
# Requiere modelo y dataset (seccion 0). Sin ellos se salta y el notebook sigue.
if not (HAY_DATASET and HAY_MODELO):
    print("Se salta: hacen falta el modelo y el dataset de repeticiones.")
else:
    # Serie sintetica a partir de las repeticiones reales de un sujeto.
    sujeto_demo = df_reps.sujeto.iloc[0]
    serie = df_reps[(df_reps.sujeto == sujeto_demo) & (df_reps.vista == "frontal")]

    if len(serie):
        etiquetas = serie.etiqueta.tolist()
        errores = [e for e in etiquetas if e != "correcto"]
        dominante = (max(set(errores), key=errores.count)
                     if len(errores) >= max(1, len(etiquetas) / 3) else "correcto")
        resumen_demo = {
            "repeticiones": len(etiquetas),
            "correctas": sum(1 for e in etiquetas if e == "correcto"),
            "clasificacion": dominante,
            "por_repeticion": etiquetas,
            "rom_max": float(serie.rom_max.max()),
            "rom_medio": float(serie.rom_max.mean()),
            "lado": serie.lado.iloc[0],
        }
        ficha = recomendacion(dominante)
        print(f"Sujeto {sujeto_demo}: {resumen_demo['repeticiones']} repeticiones")
        print(f"  secuencia: {' -> '.join(etiquetas)}")
        print(f"  dominante: {dominante}\n")
        print("TEXTO BASE (json):")
        print(f"  {ficha['recomendacion']}\n")
        if redactor.disponible:
            print("TEXTO REDACTADO (gemini):")
            texto = redactor.resumen(resumen_demo, ficha["recomendacion"])
            print(f"  {texto or '(sin respuesta)'}")


Sujeto PM_000: 4 repeticiones
  secuencia: compensacion_tronco -> correcto -> correcto -> compensacion_tronco
  dominante: compensacion_tronco

TEXTO BASE (json):
  Mantén el torso vertical, activa suavemente el abdomen y reduce el rango si necesitas inclinarte para elevar el brazo.

TEXTO REDACTADO (gemini):
  Has completado cuatro repeticiones con tu brazo derecho, logrando mantener una buena postura en el centro de la serie aunque al final ha aparecido un poco de cansancio. Para la próxima vez, concéntrate en mantener el torso completamente vertical activando suavemente el abdomen, y reduce ligeramente el movimiento si notas que necesitas inclinarte para elevar el brazo.


### 5.3 Revision de seguridad de los textos generados

Antes de poner esto delante de un paciente conviene comprobar que el modelo se
cine a las reglas. Se generan varias redacciones de la misma repeticion y se
buscan senales de que se ha salido del guion: terminos diagnosticos, indicaciones
de dosis o promesas de recuperacion.

Es una comprobacion de humo, no una garantia. Si vas a usarlo con pacientes
reales, revisa una muestra a mano.

In [17]:
# Requiere modelo y dataset (seccion 0). Sin ellos se salta y el notebook sigue.
if not (HAY_DATASET and HAY_MODELO):
    print("Se salta: hacen falta el modelo y el dataset de repeticiones.")
else:
    # La consigna debe ser corta e imperativa; ademas no puede salirse del guion.
    TERMINOS_PROHIBIDOS = [
        # diagnostico
        "tendinitis", "bursitis", "desgarro", "lesion", "artrosis", "sindrome",
        "patologia", "diagnostic", "inflamacion",
        # dosis y prescripcion
        "series", "kilos", "kg", "peso", "mancuerna", "banda elastica",
        "veces al dia", "tres veces", "descansa 48",
        # promesas
        "te curaras", "en dos semanas", "garantiza", "desaparecera",
    ]

    if redactor.disponible and ejemplos:
        clase, fila = ejemplos[-1]
        ficha = recomendacion(clase)
        print(f"Generando 5 redacciones de la misma repeticion ({clase})...\n")
        hallazgos = 0
        for i in range(5):
            texto = redactor.repeticion(
                indice=1, etiqueta=clase, confianza=0.88,
                recomendacion=ficha["recomendacion"],
                variables=fila.to_dict(), lado=fila.lado)
            if not texto:
                continue
            encontrados = [t for t in TERMINOS_PROHIBIDOS if t in texto.lower()]
            marca = "!!" if encontrados else "OK"
            hallazgos += len(encontrados)
            print(f"{marca} {texto}")
            if encontrados:
                print(f"     terminos fuera de guion: {encontrados}")
            print(f"     ({len(texto.split())} palabras)")
        print(f"\nTotal de terminos fuera de guion: {hallazgos}")
        if hallazgos:
            print("Endurece INSTRUCCION_SISTEMA en la seccion 5 y repite.")
    else:
        print("Sin Gemini disponible: nada que revisar.")

Generando 5 redacciones de la misma repeticion (rango_insuficiente)...

OK Sube más el brazo
     (4 palabras)
OK Sube más el brazo
     (4 palabras)
OK Sube más el brazo
     (4 palabras)
OK Eleva el brazo despacio.
     (4 palabras)
OK Sube más el brazo
     (4 palabras)

Total de terminos fuera de guion: 0


### 5.4 Coste y latencia

Una llamada por repeticion tiene coste. Con series de 15-20 repeticiones y varios
pacientes al dia conviene tener el numero delante antes de desplegar.

En la app la llamada de repeticion es **asincrona**: el mensaje del JSON aparece
al instante y el redactado lo sustituye cuando llega. Si tarda mas que el timeout
o falla, el paciente ve el texto base y no nota nada roto.

In [18]:
# Requiere modelo y dataset (seccion 0). Sin ellos se salta y el notebook sigue.
if not (HAY_DATASET and HAY_MODELO):
    print("Se salta: hacen falta el modelo y el dataset de repeticiones.")
else:
    import time

    if redactor.disponible and ejemplos:
        clase, fila = ejemplos[0]
        ficha = recomendacion(clase)
        tiempos = []
        for _ in range(3):
            t0 = time.time()
            redactor.repeticion(
                indice=1, etiqueta=clase, confianza=0.9,
                recomendacion=ficha["recomendacion"],
                variables=fila.to_dict(), lado=fila.lado)
            tiempos.append(time.time() - t0)
        print(f"Latencia por repeticion: {np.mean(tiempos):.2f}s "
              f"(min {min(tiempos):.2f}, max {max(tiempos):.2f})")
        print(f"Una serie de 15 repeticiones = 15 llamadas + 1 de resumen")
        reps_totales = len(df_reps)
        print(f"Reprocesar las {reps_totales} repeticiones de este dataset serian "
              f"{reps_totales} llamadas (~{reps_totales * np.mean(tiempos) / 60:.0f} min)")
    else:
        print("Sin Gemini disponible: no se puede medir la latencia.")

Latencia por repeticion: 0.51s (min 0.44, max 0.61)
Una serie de 15 repeticiones = 15 llamadas + 1 de resumen
Reprocesar las 334 repeticiones de este dataset serian 334 llamadas (~3 min)


---
## 6. Voz de las consignas con Google Cloud Text-to-Speech

Quien eleva el brazo tiene la vista en su propio hombro, no en la pantalla. El
rotulo sobre el video ayuda, pero una consigna **dicha en voz alta** es lo que de
verdad se parece a tener un fisioterapeuta al lado.

### El diseno esta centrado en la cache, y por una razon concreta

Las consignas son un **conjunto cerrado y pequeno**: las de
`knowledge_base/ejercicios.json` son diez en total, y se repiten en cada serie de
cada paciente. Sintetizarlas una vez y guardarlas en disco convierte la
reproduccion en una lectura de archivo: latencia nula, coste nulo y funciona sin
red. Solo las consignas que redacta Gemini, que varian, pueden provocar una
sintesis nueva — y tambien se cachean.

Esta celda precalienta esa cache. Ejecutarla una vez deja la app lista.

### Autenticacion: no vale la clave de Gemini

Verificado contra el servicio: Cloud TTS responde
`401 UNAUTHENTICATED — API keys are not supported by this API`. Necesita
credenciales de **cuenta de servicio**, que es un alta distinta:

1. En la consola de Google Cloud, habilita *Cloud Text-to-Speech API*.
2. Crea una cuenta de servicio y descarga su JSON.
3. Anade al `.env` la ruta de ese archivo:

```
GOOGLE_APPLICATION_CREDENTIALS=/ruta/a/credenciales.json
```

Sin esto todo funciona igual, pero en silencio.

In [19]:
# ---------------------------------------------------------------------------
# Voz, en su version minima
#
# El diseno esta centrado en la cache y por una razon concreta: las consignas
# son un conjunto cerrado y pequeno —diez en la base de conocimiento— que se
# repiten en cada serie. Sintetizarlas una vez convierte la reproduccion en una
# lectura de archivo: latencia nula, coste nulo y funciona sin red.
#
# Se usa el motor de Gemini, que vale con la misma clave de API. Cloud TTS
# necesita una cuenta de servicio, que en Colab es un archivo mas que montar.
# ---------------------------------------------------------------------------
import hashlib
import wave

DIR_AUDIO = DATOS_DIR / "audio"
DIR_AUDIO.mkdir(parents=True, exist_ok=True)
VOZ_GEMINI = os.environ.get("PHYSIOVISION_TTS_VOZ_GEMINI", "Kore")


def ruta_audio(texto: str) -> Path:
    """Un archivo por texto. El nombre es el resumen del texto, no el texto."""
    firma = hashlib.sha256(texto.strip().lower().encode("utf-8")).hexdigest()[:16]
    return DIR_AUDIO / f"nb_{firma}.wav"


def sintetizar(texto: str) -> Path | None:
    """Devuelve la ruta del audio. None si no se puede. Nunca lanza."""
    texto = (texto or "").strip()
    if not texto:
        return None
    destino = ruta_audio(texto)
    if destino.exists():
        return destino
    if not redactor.disponible:
        return None
    try:
        respuesta = redactor.cliente.models.generate_content(
            model="gemini-2.5-flash-preview-tts",
            contents=texto,
            config=redactor.genai.types.GenerateContentConfig(
                response_modalities=["AUDIO"],
                speech_config=redactor.genai.types.SpeechConfig(
                    voice_config=redactor.genai.types.VoiceConfig(
                        prebuilt_voice_config=redactor.genai.types.PrebuiltVoiceConfig(
                            voice_name=VOZ_GEMINI)))),
        )
        parte = respuesta.candidates[0].content.parts[0].inline_data
        # La API devuelve PCM crudo; se envuelve en WAV para poder reproducirlo.
        with wave.open(str(destino), "wb") as f:
            f.setnchannels(1); f.setsampwidth(2); f.setframerate(24000)
            f.writeframes(parte.data)
        return destino
    except Exception as error:
        print(f"  (sin voz: {type(error).__name__})")
        return None


CONSIGNAS = sorted({c for datos in CONOCIMIENTO.values()
                    for info in datos.get("errores", {}).values()
                    for c in info.get("consignas", [])})
print(f"{len(CONSIGNAS)} consignas en la base de conocimiento:")
for c in CONSIGNAS:
    print(f"  [{'cacheada ' if ruta_audio(c).exists() else 'pendiente'}] {c}")


10 consignas en la base de conocimiento:
  [pendiente] Activa el abdomen
  [pendiente] Busca más recorrido
  [pendiente] Eleva un poco más
  [pendiente] Mantén el torso recto
  [pendiente] No inclines el cuerpo
  [pendiente] Sube más el brazo
  [pendiente] ¡Así es!
  [pendiente] ¡Bien hecho!
  [pendiente] ¡Correcto!
  [pendiente] ¡Muy bien!


In [20]:
# Precalentado. Idempotente: lo ya sintetizado no se vuelve a pedir.
hechas = [c for c in CONSIGNAS if ruta_audio(c).exists()]
pendientes = [c for c in CONSIGNAS if not ruta_audio(c).exists()]
if pendientes and redactor.disponible:
    print(f"Sintetizando {len(pendientes)} consignas...")
    for texto in pendientes:
        if sintetizar(texto):
            hechas.append(texto)
print(f"{len(hechas)}/{len(CONSIGNAS)} consignas con audio en {DIR_AUDIO}")


Sintetizando 10 consignas...
  (sin voz: AttributeError)
  (sin voz: ClientError)
  (sin voz: ClientError)
  (sin voz: ClientError)
  (sin voz: ClientError)
5/10 consignas con audio en /Users/dafnezepedagonzalez/Documents/Diplomado/Modulo5/physiovision_gradio/data/audio


### 6.1 Escuchar el resultado

Si hay audio, esta celda lo reproduce dentro del notebook. Merece la pena oirlo
antes de ponerlo delante de un paciente: la velocidad y la voz cambian bastante
la sensacion, y ambas se ajustan por variable de entorno
(`PHYSIOVISION_TTS_VELOCIDAD`, `PHYSIOVISION_TTS_VOZ`).

In [21]:
from IPython.display import Audio, display

reproducidas = 0
for texto in CONSIGNAS:
    ruta = ruta_audio(texto)
    if not ruta.exists():
        continue
    print(f"{texto}   ({ruta.stat().st_size / 1024:.0f} KB)")
    display(Audio(str(ruta)))
    reproducidas += 1

if not reproducidas:
    print("Todavia no hay audio en la cache. Ejecuta la celda anterior con")
    print("GEMINI_API_KEY configurada.")


Busca más recorrido   (92 KB)


Eleva un poco más   (77 KB)


Mantén el torso recto   (98 KB)


No inclines el cuerpo   (100 KB)


Sube más el brazo   (85 KB)


### 6.2 Coste

El precalentado son diez sintesis, una sola vez. A partir de ahi una sesion de
quince repeticiones no hace **ninguna** llamada a Cloud TTS, porque todas las
consignas salen de la cache.

La excepcion son las consignas redactadas por Gemini, que varian y provocan una
sintesis la primera vez que aparece cada texto. Aun asi el vocabulario converge
rapido: son variaciones sobre las mismas tres ideas.

In [22]:
cacheadas = [c for c in CONSIGNAS if ruta_audio(c).exists()]
peso = sum(f.stat().st_size for f in DIR_AUDIO.glob("nb_*.wav"))
print(f"consignas fijas cacheadas : {len(cacheadas)}/{len(CONSIGNAS)}")
print(f"peso de la cache          : {peso / 1024:.0f} KB")
print(f"llamadas por serie de 15  : "
      f"{0 if len(cacheadas) == len(CONSIGNAS) else 'depende'}"
      " (las consignas fijas ya no se sintetizan)")
print("\nLas que varian son las que redacta Gemini; aun asi el vocabulario")
print("converge rapido: son variaciones sobre las mismas tres ideas.")


consignas fijas cacheadas : 5/10
peso de la cache          : 453 KB
llamadas por serie de 15  : depende (las consignas fijas ya no se sintetizan)

Las que varian son las que redacta Gemini; aun asi el vocabulario
converge rapido: son variaciones sobre las mismas tres ideas.


---
## 7. PhysioVision en Gradio, con la camara

La misma sesion en vivo de la seccion 4, ya con todo lo demas encima: el
fotograma entra, se dibuja el esqueleto, y **cuando una repeticion se cierra
aparece su veredicto, Gemini redacta la consigna y suena en voz alta**. Es el
reparto de la aplicacion: los angulos se mueven todo el rato, el juicio llega
una vez por repeticion.

Cuatro detalles que no son cosmeticos:

- **Se piden 30 Hz** (`stream_every=1/30`). No es comodidad: el modelo se
  entreno a esa tasa y `suavidad_ldlj` es una derivada tercera, que se desplaza
  con el muestreo. Medido: a 10 Hz una repeticion de cada cuatro cambia de
  clase. Pero es un **techo**, no una promesa: el frontend no captura el
  fotograma siguiente hasta que vuelve el anterior, asi que la tasa real la fija
  el viaje de ida y vuelta. Por eso la interfaz la muestra en pantalla.
- **La camara se abre a 640x480** (`RESOLUCION_CAMARA`). Sin acotarla, el
  navegador la abre a su tamano nativo y cada fotograma viaja entero: 121 KB por
  fotograma en lugar de 52 KB, medido sobre un fotograma real. Esa es la palanca
  que sube los fps, no bajar `stream_every`.
- **Gemini y la voz no bloquean la camara**. Al cerrarse una repeticion se
  muestra en el acto la consigna de la base de conocimiento y el trabajo lento
  se encarga a un hilo aparte. Hacerlo en linea congelaria la imagen uno o dos
  segundos en cada repeticion.
- **La imagen no se refleja**. Reflejarla intercambiaria izquierda y derecha, y
  el seguimiento acabaria en el brazo equivocado.

Diferencias con la aplicacion completa (`python app.py`), que aqui sobran:
historial en base de datos, autenticacion, pausa y las vistas de progreso.


### 7.1 El callback: un fotograma dentro, una repeticion fuera

La regla que gobierna esta celda: **el manejador del stream no espera a nadie**.
Todo lo que tarde —Gemini, la sintesis de voz— sale a un hilo aparte y se recoge
en un fotograma posterior, porque mientras este manejador no vuelve, la camara
esta parada.


In [23]:
# Requiere el modelo entrenado (seccion 0). Sin ello se salta y el notebook sigue.
if not HAY_MODELO:
    print("Se salta: falta el modelo entrenado.")
else:
    import threading
    from concurrent.futures import ThreadPoolExecutor

    # -----------------------------------------------------------------------
    # Redaccion y voz, fuera del hilo de la camara
    #
    # Gemini tarda uno o dos segundos, y la sintesis de voz otro tanto. Hacerlo
    # dentro del manejador del stream congela la imagen justo cuando el paciente
    # acaba de moverse: el frontend de Gradio no captura el fotograma siguiente
    # hasta que vuelve el anterior, asi que esos segundos son camara detenida.
    #
    # El reparto correcto es el de la aplicacion: al cerrarse la repeticion se
    # muestra YA la consigna de la base de conocimiento, que esta en memoria, y
    # el trabajo lento se encarga a un hilo aparte. Cuando termina, un fotograma
    # posterior lo recoge y sustituye la tarjeta. Si Gemini falla o tarda, el
    # paciente ve el texto base y no nota nada roto.
    # -----------------------------------------------------------------------
    class Enriquecedor:
        """Cola de un hilo para el texto y el audio de cada repeticion."""

        def __init__(self) -> None:
            self._pool = ThreadPoolExecutor(max_workers=1,
                                            thread_name_prefix="pv-texto")
            self._lock = threading.Lock()
            self._listo: tuple[str, str | None] | None = None

        def encargar(self, resultado: dict, lado: str | None) -> None:
            """Lanza el trabajo lento y vuelve en el acto."""

            def tarea() -> None:
                texto = resultado["consigna"]
                if redactor.disponible:
                    ficha = recomendacion(resultado["label"])
                    texto = redactor.repeticion(
                        indice=resultado["indice"], etiqueta=resultado["label"],
                        confianza=resultado["confidence"],
                        recomendacion=ficha["recomendacion"],
                        variables=resultado["variables"], lado=lado) or texto
                audio = sintetizar(texto)
                if texto == resultado["consigna"] and audio is None:
                    return       # nada que anadir: la tarjeta ya dice eso mismo
                with self._lock:
                    self._listo = (tarjeta_md(resultado, texto),
                                   str(audio) if audio else None)

            try:
                self._pool.submit(tarea)
            except RuntimeError:
                pass             # el pool ya estaba cerrado: no es un error

        def recoger(self) -> tuple[str, str | None] | None:
            """Devuelve `(tarjeta, audio)` si el hilo trajo algo, o `None`."""
            with self._lock:
                listo, self._listo = self._listo, None
            return listo

        def cerrar(self) -> None:
            self._pool.shutdown(wait=False, cancel_futures=True)


    def tarjeta_md(resultado: dict, texto: str) -> str:
        """La tarjeta del veredicto, con el texto que haya disponible."""
        v = resultado["variables"]
        marca = "✅" if resultado["label"] == "correcto" else "⚠️"
        return (f"## {marca} Repeticion {resultado['indice']}: "
                f"{resultado['label']}\n\n### {texto}\n\n"
                f"<sub>rango {v['rom_max']:.0f} grados · "
                f"tronco {v['tronco_max']:.0f} · {v['duracion_s']:.1f} s · "
                f"confianza {resultado['confidence']:.0%}</sub>")


    def iniciar_sesion_ui(brazo: str):
        """Crea la sesion. Vive en el estado del navegador, no en una global:
        el detector de MediaPipe tiene estado y no es seguro entre hilos."""
        lado = brazo if brazo in ("left", "right") else None
        return (SesionNotebook(fps=30.0, lado=lado), Enriquecedor(),
                "Sesion iniciada. Colocate de cuerpo entero frente a la camara.",
                None, None)

    def procesar_ui(frame, sesion, enriquecedor):
        """Se ejecuta en cada fotograma que llega, y **nunca espera a la red**:
        al cerrarse una repeticion solo encarga el texto y el audio."""
        if sesion is None or frame is None:
            return frame, gr.skip(), gr.skip(), gr.skip(), sesion, enriquecedor

        anotado, metricas, resultado = sesion.procesar(frame)
        pantalla = para_pantalla(anotado)

        if metricas is None:
            return (pantalla, "Sin persona en el encuadre", gr.skip(), gr.skip(),
                    sesion, enriquecedor)

        # La tasa real, a la vista: si cae muy por debajo de los 30 Hz del
        # entrenamiento, las duraciones de cada repeticion se miden mal.
        tasa = f" · {sesion.fps_real:.0f} fps"
        if metricas.calibrando:
            estado = "Calibrando: detectando que brazo trabaja..." + tasa
        else:
            estado = (f"**{metricas.fase}** · hombro {metricas.abduccion:.0f} grados"
                      f" · tronco {metricas.inclinacion_tronco:.0f}"
                      f" · repeticiones {metricas.repeticiones}{tasa}")

        if resultado is not None:
            # Repeticion cerrada: el veredicto se ve en el acto, con la consigna
            # de la base de conocimiento. Gemini y la voz van por detras.
            enriquecedor.encargar(resultado, sesion.acumulador.lado)
            return (pantalla, estado, tarjeta_md(resultado, resultado["consigna"]),
                    gr.skip(), sesion, enriquecedor)

        listo = enriquecedor.recoger() if enriquecedor is not None else None
        if listo is not None:
            tarjeta, audio = listo
            return (pantalla, estado, tarjeta, audio or gr.skip(),
                    sesion, enriquecedor)

        return pantalla, estado, gr.skip(), gr.skip(), sesion, enriquecedor

    def terminar_ui(sesion, enriquecedor):
        """Cierra la serie y devuelve el resumen redactado.

        Aqui si se puede esperar a Gemini: la camara ya no esta en marcha.
        """
        if enriquecedor is not None:
            enriquecedor.cerrar()
        if sesion is None:
            return None, None, "No hay ninguna sesion activa.", gr.skip(), gr.skip()
        resumen = sesion.resumen()
        sesion.cerrar()
        if resumen["repeticiones"] == 0:
            return (None, None, "No se detecto ninguna repeticion completa.",
                    gr.skip(), gr.skip())

        ficha = recomendacion(resumen["clasificacion"])
        texto = ficha["recomendacion"]
        if redactor.disponible:
            texto = redactor.resumen(resumen, texto) or texto
        audio = sintetizar(texto)
        cierre = (f"## {ficha['titulo']}\n\n"
                  f"**{resumen['correctas']}/{resumen['repeticiones']} correctas** · "
                  f"rango maximo {resumen['rom_max']:.0f} grados\n\n{texto}\n\n"
                  f"> {ficha['precaucion']}")
        return None, None, cierre, gr.skip(), (str(audio) if audio else gr.skip())

    print("callbacks de la sesion en vivo definidos")


callbacks de la sesion en vivo definidos


### 7.2 La interfaz

In [24]:
import gradio as gr

ESTADO = (f"modelo (macro-F1 {CONTRATO['metricas_loso']['macro_f1']:.2f}) · "
          f"Gemini {'disponible' if redactor.disponible else 'no disponible'} · "
          f"serie de {REPS_OBJETIVO} repeticiones")

with gr.Blocks(title="PhysioVision — camara") as demo:
    gr.Markdown(f"""
    # PhysioVision — elevacion lateral de hombro
    Pulsa **Iniciar** y colocate de cuerpo entero frente a la camara. El
    resultado aparece **al cerrarse cada repeticion**, no al final.

    <sub>{ESTADO}</sub>
    """)

    # El detector de MediaPipe tiene estado: una sesion por navegador, nunca
    # una global compartida. Y con ella, la cola que redacta y sintetiza por
    # detras, que tambien es de este navegador y de nadie mas.
    sesion_ui = gr.State(None)
    enriquecedor_ui = gr.State(None)

    with gr.Row():
        brazo = gr.Dropdown(label="Brazo",
                            choices=[("Detectar solo", "auto"), ("Derecho", "right"),
                                     ("Izquierdo", "left")],
                            value="right", scale=1,
                            info="Indicarlo mejora la deteccion de repeticiones.")
        boton_iniciar = gr.Button("Iniciar", variant="primary", scale=0)
        boton_terminar = gr.Button("Terminar serie", scale=0)

    with gr.Row():
        with gr.Column(scale=3):
            camara = gr.Image(label="Camara", sources=["webcam"], streaming=True,
                              type="numpy",
                              # Sin espejo: reflejar intercambiaria izquierda y
                              # derecha y el seguimiento iria al brazo erroneo.
                              # Y con la resolucion acotada: sin `constraints`
                              # el navegador abre la camara a su tamano nativo y
                              # cada fotograma viaja entero, que es lo que hunde
                              # los fps. Ver RESOLUCION_CAMARA.
                              webcam_options=gr.WebcamOptions(
                                  mirror=False, constraints=RESOLUCION_CAMARA))
            salida = gr.Image(label="Seguimiento", interactive=False)
        with gr.Column(scale=2):
            estado = gr.Markdown("Sin sesion activa")
            tarjeta = gr.Markdown("El veredicto aparece al cerrar la primera "
                                  "repeticion.")
            voz_ui = gr.Audio(label="Consigna", autoplay=True, interactive=False)

    boton_iniciar.click(iniciar_sesion_ui, [brazo],
                        [sesion_ui, enriquecedor_ui, estado, tarjeta, voz_ui])
    boton_terminar.click(terminar_ui, [sesion_ui, enriquecedor_ui],
                         [sesion_ui, enriquecedor_ui, tarjeta, estado, voz_ui])
    camara.stream(procesar_ui, [camara, sesion_ui, enriquecedor_ui],
                  [salida, estado, tarjeta, voz_ui, sesion_ui, enriquecedor_ui],
                  # 30 Hz es la tasa a la que se entreno el modelo, pero aqui es
                  # un techo, no una promesa: la tasa real la fija el viaje de
                  # ida y vuelta de cada fotograma.
                  stream_every=1 / 30, concurrency_limit=2,
                  show_progress="hidden")

print("Interfaz construida. La celda siguiente la levanta.")


Interfaz construida. La celda siguiente la levanta.


### 7.3 Levantar la app

In [27]:
# Se levanta dentro del notebook. En Colab no hay navegador local al que
# asomarse, asi que se pide un enlace publico temporal; en local, no.
# Para pararla, ejecuta la celda siguiente.
demo.queue(default_concurrency_limit=2).launch(
    inline=not EN_COLAB, share=EN_COLAB, quiet=True, height=900)


### 7.4 Parar el servidor

In [26]:
demo.close()
print("Servidor detenido. La app completa se levanta con:  python app.py")

Closing server running on port: 7860
Servidor detenido. La app completa se levanta con:  python app.py


---
## 8. De aqui a la app real

Lo que acaba de correr consume exactamente los artefactos del primer notebook. Para
llevarlo a la aplicacion que se usa de verdad:

```bash
python app.py            # camara en vivo, historial y sesiones
```

La app hace lo mismo que la seccion 7 de este notebook, con las piezas repartidas
en modulos y con lo que aqui sobraba: historial en base de datos, autenticacion,
pausa y vistas de progreso. El nucleo es identico —mismo contrato, mismas
variables, misma segmentacion causal— y esa es la razon de haberlo copiado aqui
en vez de importarlo: si las dos copias divergen, divergen las predicciones.

### Notas importantes

1. **Vista lateral.** Aqui se asume frontal (`es_lateral = 0`). El modelo tiene la
   variable y el dataset tiene las dos vistas: falta decidir en la interfaz cual es.
2. **Cambio de dominio.** El modelo se entreno con camaras fijas de laboratorio y la
   app recibe video de movil. El rendimiento real sera menor que el macro-F1 del
   contrato.
3. **Latencia de Gemini.** En esta version las llamadas son sincronas y una serie de
   quince repeticiones se nota. La app las hace en segundo plano y muestra primero el
   texto del JSON.
4. **Etiquetas.** Si el contrato dice `etiquetado: reglas_automaticas`, las metricas
   miden consistencia interna, no validez clinica. Revisa
   `data/datasets/ex1_etiquetas.csv` y reentrena.